In [ ]:
# Solar Filament V3.0 | one-cell Kaggle entry
# Add competition data and the GitHub filament-seg-weights Dataset.
# Modes: preflight, train, smoke, eval, submit, full.
# The local Blended architecture is enabled for shared-split retraining.
# Its exact V10 identity remains unverified. Enable Internet for training.
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

CONFIG = {
    'mode': 'full',
    'input_root': '/kaggle/input',
    'working_root': '/kaggle/working',
    'scratch_root': '/kaggle/temp/solar_v3',
    'blended_checkpoint': '',
    'accept_local_blended_source': True,
    'exact_v10_source': False,
    'source_sha256': '3f3815d5f2b3be9ce7604bc982ee6c87c6ecc18f84cf23dfd79973769a3cd830',
    'source_name': 'blended_local.txt',
    'train_batch': 4,
    'mosaic_batch': 2,
    'export_all_maps': False,
    'install_missing': True,
}

MODULES = {
    'blended_source.py': """\"\"\"
Solar Filament Segmentation Challenge 2026 -- mosaic (tiled) baseline.

Binary segmentation (filament vs background) at NATIVE resolution:
  - COCO polygons (all 4 categories) merged into ONE foreground mask; the several
    annotator passes that share a file_name are unioned.
  - CENTRED CROP. Only the central 1872x1872 of each 2048^2 frame is modelled: the
    corners are black, the disk has radius ~1005px, and no annotated filament reaches
    past 903px from the centre, so the crop keeps margin while dividing evenly.
  - MOSAIC TILING with OVERLAP. A 2x2 grid of 1056x1056 tiles is laid across the crop
    with the first and last flush against its edges, so neighbours share a 240px band
    and every tile sits wholly inside the crop -- the pixels a tile sees are always
    real, never reflected or padded. The whole tile is supervised.
  - THE OVERLAP IS BLENDED, not cut. A pixel two or four tiles predicted becomes the
    weighted mean of those predictions, each weighted by how far the pixel sits from
    its own tile's edge, so the tile that saw it centrally outvotes the one that only
    caught it at the rim, and no seam is left for the instance labelling to trip on.
  - Cache: 2048^2 uint8 memmaps for images and masks, plus a list of foreground pixels
    per image. Filaments cover ~0.2% of pixels, so uniformly random crops would be
    mostly empty -- FG_PROB of them are anchored on a filament pixel.
  - U-Net: any timm encoder (ImageNet, in_chans=1) + a decoder driven by the
    encoder's stage reductions.
  - Loss: BCE(pos_weight) + soft Dice over the whole tile. bf16 autocast. Two
    metric-shaped variants (clDice + instance-balanced pixel weights, and a
    per-instance soft Dice) both submitted worse and have been removed: the metric
    being instance-level does not mean a loss shaped like it trains better.
  - Validation reports what the submission is judged on: the DISTRIBUTIONS of
    per-instance Dice and IoU, per-image Dice, and the one-to-many / many-to-one
    relation counts between ground truth and prediction. It runs through the exact
    post-processing the submission uses, and both the kept weights and the decision
    threshold are chosen by it. The raw distributions from the best epoch land in
    checkpoints/val_distributions.csv, ready to plot for the report.
  - AdamW with per-part LRs (encoder / bottleneck / decoder / head) + one cosine curve.
  - No checkpoint is written. The best epoch's weights are held in RAM and, once
    training ends, predict the test set straight into submission.csv -- so what gets
    submitted is exactly the state validation picked, with no file in between.

Everything is hardcoded below. Just: python train.py
\"\"\"

import json
import math
import os
import random
import time
from collections import defaultdict

import albumentations as A
import cv2
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from pycocotools import mask as coco_mask
from torch.utils.data import Dataset
from tqdm.auto import tqdm

# --------------------------------------------------------------------------- #
# config
# --------------------------------------------------------------------------- #
DATA = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
WORK_DIR = "/kaggle/working"
CACHE_DIR = "/kaggle/temp/cache"   # scratch, so the ~6 GB cache is not committed as output

TRAIN_DIR = os.path.join(DATA, "train", "train_images")
TEST_DIR = os.path.join(DATA, "test", "test_images")
ANN_JSON = os.path.join(DATA, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")  # only the val distributions land here
OUT_CSV = os.path.join(WORK_DIR, "submission.csv")


# --- mosaic geometry -------------------------------------------------------- #
# The solar disk (radius ~1005px) sits centred in the 2048^2 frame, the corners are
# literally black, and the furthest annotated filament vertex is 903px from the centre
# (measured over every polygon in the train JSON), so only a centred CROP is modelled.
#
# TILES x TILES tiles are spread across the crop with the first and last flush against
# its edges, so consecutive tiles OVERLAP by OVERLAP px. Every tile lies entirely inside
# the crop, which is what makes the context real: a tile never reaches past the frame,
# so nothing it sees is reflected or padded. Unlike a core/halo mosaic there is no
# unsupervised strip -- the whole tile carries loss -- and no hard cut at reassembly.
#
# Two constraints, not five. TILE must land on the encoder's stride-32 grid or
# UNet.forward has to reflect-pad internally (1056 = 33*32), and the tiles must actually
# overlap. CROP is otherwise free: overlapping tiles impose no divisibility, so it is
# set purely by what has to be covered -- half-width 936 clears the furthest annotated
# filament vertex (903px from the centre) by 33px.
FULL = 2048                     # native image size / submission RLE size
CROP = 1872                     # centred region that gets predicted
OFFSET = (FULL - CROP) // 2     # 88
# TILE follows from TILES: with the first and last tile flush against the crop,
# OVERLAP = (TILES*TILE - CROP) / (TILES - 1). The two cannot be set independently --
# holding TILE fixed while changing TILES swings the overlap wildly (736px gives 23%
# on a 3x3 grid and 49% on a 4x4 one). 1056 holds the overlap at 23% for a 2x2 grid.
TILE = 1056                     # 33*32, so the encoder needs no internal padding
TILES = 2                       # tiles per axis -> 2x2 = 4 forward passes per image
STEP = (CROP - TILE) // (TILES - 1)   # 816px between neighbouring tile origins
OVERLAP = TILE - STEP                 # 240px shared by each neighbouring pair
BLEND_FLOOR = 0.05              # blend weight at a tile's outermost pixel; never 0
TTA = 8                         # dihedral transforms averaged at inference:
                                # 1 = off, 4 = the rot90s, 8 = those plus a flip
assert TILE % 32 == 0, "TILE must sit on the encoder's stride-32 grid"
assert TILE <= CROP, "a tile has to fit inside the crop"
assert TILES >= 2 and OVERLAP > 0, "tiles must overlap -- raise TILES or TILE"
assert (FULL - CROP) % 2 == 0, "CROP must be centrable in FULL"

ENCODER = "efficientvit_b2.r288_in1k"
DECODER_CH = (192, 96, 48, 24, 12)
BOTTLENECK_CH = 192     # width inside the encoder->decoder bottleneck (0 = no bottleneck)
BOTTLENECK_DIL = (1, 2, 4)   # dilations of its 3x3 convs, widening context at /32

EPOCHS = 24
BATCH_SIZE = 4          # tiles per step. Measured 9.5 GB at 1056^2 in fp16, against
                        # 16 GB on a T4/P100. A 2x2 grid needs a big tile, and a big
                        # tile caps the batch -- 4 is small for the encoder's
                        # BatchNorm, which is the real cost of this layout.
CROPS_PER_IMAGE = 4     # random tiles drawn per training image per epoch. Per-epoch
                        # coverage is CROPS_PER_IMAGE * TILE^2 / CROP^2 = 62%, up from
                        # 44% under core/halo: the whole tile counts now, not its core.
FG_PROB = 0.7           # share of crops anchored on a filament pixel
FG_POINTS = 128         # foreground pixels cached per image to anchor crops
TILE_BATCH = 4          # tiles per forward pass when assembling a mosaic (= all 4)
# per-part learning rates: the encoder is ImageNet-pretrained and only needs gentle
# fine-tuning, the decoder is random-init, the head is a single 1x1 logit conv
ENCODER_LR = 1e-4
DECODER_LR = 5e-4
HEAD_LR = 5e-4
BOTTLENECK_LR = 5e-4     # random-init like the decoder, so it trains at the same rate
WEIGHT_DECAY = 2e-2
WARMUP_EPOCHS = 2
# Loss: BCE_W * BCE + DICE_W * soft Dice, per sample. A PQ-shaped loss (clDice for
# connectivity, instance-balanced pixel weights) was tried and scored WORSE on
# submission -- the metric being instance-level does not mean a loss shaped like it
# trains better -- so the two terms stay generic.
#
# POS_WEIGHT is the consequential number here. It was 10, on the reasoning that
# filaments are ~0.2% of a frame; but weighting positives by w drives the network's
# output to q = w*p / (1 - p + w*p) for a true posterior p, which systematically
# inflates every probability and drags the decision threshold with it (see
# EVAL_THRESHOLDS below). At 1.0 the network is calibrated: q = p, and 0.5 means 0.5.
# The Dice term already supplies the pressure against predicting empty, which is what
# POS_WEIGHT was really there for. Set it back to 10.0 to undo this in one number.
POS_WEIGHT = 1.0
BCE_W = 0.45
DICE_W = 0.55
# Smoothing added to both ends of the Dice ratio. Large values quietly forgive a tile
# that is nearly all background -- with eps=1 and a 300px filament in a 1M-pixel tile,
# the constant is a tenth of the numerator. Keep it only large enough to avoid 0/0.
DICE_EPS = 1e-6

# The cached mask is a plain 0/1 label. An older layout encoded a per-filament weight
# into the same array (small filaments weighted up, since the metric counts every
# instance once whatever its area); everything downstream only ever tested "> 0", so a
# cache written under that layout still reads correctly and needs no rebuild.
CACHE_VER = 2           # bump when the cache layout changes

# post-processing: validation and the submission run it, so val numbers are the
# submission's numbers. MIN_AREA / CLOSE_KERNEL are the values used when there is no
# validation set to search with; when there is one, evaluate() searches the whole
# EVAL_* grid below and main() submits with whatever won.
MIN_AREA = 150          # drop blobs smaller than this (0.3% of GT instances)
CLOSE_KERNEL = 5        # morphological closing to rejoin broken threads (0 = off)

# Where the optimum sits depends on POS_WEIGHT. Minimising BCE with positives weighted
# by w drives the output to q = w*p / (1 - p + w*p), so a cut at q = 0.5 is really a cut
# at the posterior p = 1/(w+1): at w = 10 that is 0.09, far too liberal, and the optimum
# gets dragged up towards q = w/(1+w) = 0.91. At w = 1 the network is calibrated and the
# optimum should sit near 0.5. The range is left wide on purpose -- it costs one extra
# labelling pass per threshold and removes the risk of the search hitting its own
# ceiling, which a range topping out at 0.7 could have been doing all along.
EVAL_THRESHOLDS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
EVAL_MIN_AREAS = (100, 150, 250, 400)
EVAL_CLOSE_KERNELS = (0, 5, 9)
# One cell per (threshold, min_area, close_kernel). Cost is one connected-components
# pass per (threshold, kernel), NOT per cell: min_area only ever drops whole components,
# so it is applied as a column mask over an already-computed overlap matrix.
EVAL_GRID = tuple((t, a, k) for t in EVAL_THRESHOLDS
                  for a in EVAL_MIN_AREAS
                  for k in EVAL_CLOSE_KERNELS)

# What the submission is judged on, so what validation reports and selects on:
#   - the DISTRIBUTION of Dice scores      (per instance, and per image)
#   - the DISTRIBUTION of IoU scores       (per instance)
#   - one-to-many / many-to-one relations  (fragmentation and over-merging)
# A prediction and a ground-truth instance count as "related" once one covers this
# fraction of the other, which is what turns raw overlap into a countable relation.
REL_OVERLAP = 0.1
PERCENTILES = (10, 25, 50, 75, 90)
NUM_WORKERS = 4
VAL_FRAC = 0.075
# VAL_FRAC = 0.0 trains on every image and skips validation entirely: no mosaic pass
# per epoch, no epoch selection, no threshold search. The last epoch is what gets
# submitted, at NO_VAL_THRESHOLD -- with no held-out set the decision threshold cannot
# be discovered, so it has to be stated. Use this only once the geometry and the
# threshold are settled and the remaining win is the extra VAL_FRAC of training data.
NO_VAL_THRESHOLD = 0.5
SEED = 42

# Per-image intensity normalization. The six GONG sites and eleven years of data
# differ in exposure, flat-fielding and seeing, so a fixed /255 leaves the encoder to
# absorb an instrument offset that has nothing to do with filaments. Each frame is
# centred on the median and scaled by the 16-84 spread of the pixels inside a centred
# NORM_RADIUS disk -- well inside the limb, since the black corners would drag the
# median down and limb darkening would inflate the spread.
NORM_RADIUS = 900       # px; every annotated filament is within 903px of the centre

DEVICE = "cuda"

# float16, not bfloat16: bf16 needs compute capability 8.0+ and Kaggle's T4 (7.5) and
# P100 (6.0) do not have it. float16 has a narrow exponent, so training runs under a
# GradScaler -- correct on every Kaggle GPU, including the ones that could have done
# bf16, at the cost of a little numerical headroom.
AMP_DTYPE = torch.float16

_yy, _xx = np.ogrid[:FULL, :FULL]
DISK_MASK = ((_yy - FULL // 2) ** 2 + (_xx - FULL // 2) ** 2) <= NORM_RADIUS ** 2


def norm_stats(img):
    \"\"\"(centre, scale) for one full frame: median and 16-84 spread inside the disk.

    Exact for uint8 via a 256-bin histogram, which is a single O(n) pass -- np.percentile
    would sort ~2.5M values per call, and this runs once per image on every val pass.
    \"\"\"
    hist = np.bincount(img[DISK_MASK], minlength=256).astype(np.float64)
    cdf = np.cumsum(hist) / max(hist.sum(), 1.0)
    lo, mid, hi = np.searchsorted(cdf, (0.16, 0.50, 0.84))
    return float(mid), float(max(hi - lo, 1.0))


def build_norm_cache(img_path, n):
    \"\"\"Per-image (centre, scale) for every cached train image, measured once.

    Kept out of build_cache so an existing image/mask cache does not have to be rebuilt
    to gain it. Training reads these; validation and the submission recompute the same
    numbers from the frame in hand, so the two paths agree by construction.
    \"\"\"
    path = os.path.join(CACHE_DIR, f"norm_{FULL}_v{CACHE_VER}.npy")
    if os.path.exists(path):
        return np.load(path)
    images = np.load(img_path, mmap_mode="r")
    out = np.zeros((n, 2), dtype=np.float32)
    for i in tqdm(range(n), desc="[cache] intensity stats", unit="img"):
        out[i] = norm_stats(np.asarray(images[i]))
    np.save(path, out)
    return out

# --------------------------------------------------------------------------- #
# cache building
# --------------------------------------------------------------------------- #
def build_cache():
    \"\"\"Decode every train JPEG + rasterize its polygons once, at native resolution.

    The FULL frame is cached, not the crop: augmentation reads a WORK-wide window
    around each tile, which reaches past the crop into the border band. All cached
    coordinates are in full-frame space.

    Returns (img_path, msk_path, fg_path, cnt_path, names). The two big arrays are
    ~2.6 GB each; they are written as .npy memmaps and read lazily afterwards.
    \"\"\"
    os.makedirs(CACHE_DIR, exist_ok=True)
    img_path = os.path.join(CACHE_DIR, f"images_{FULL}_v{CACHE_VER}.npy")
    msk_path = os.path.join(CACHE_DIR, f"masks_{FULL}_v{CACHE_VER}.npy")
    fg_path = os.path.join(CACHE_DIR, f"fgpts_{FULL}_v{CACHE_VER}.npy")
    cnt_path = os.path.join(CACHE_DIR, f"fgcnt_{FULL}_v{CACHE_VER}.npy")
    names_path = os.path.join(CACHE_DIR, f"names_{FULL}_v{CACHE_VER}.json")

    if all(os.path.exists(p) for p in (img_path, msk_path, fg_path, cnt_path, names_path)):
        names = json.load(open(names_path))
        print(f"[cache] reusing {len(names)} cached images at {FULL}px")
        return img_path, msk_path, fg_path, cnt_path, names

    ann = json.load(open(ANN_JSON))

    # image_id -> file_name  (several image_ids map to the same file: annotator passes)
    id2file = {im["id"]: im["file_name"] for im in ann["images"]}
    id2hw = {im["id"]: (im["height"], im["width"]) for im in ann["images"]}

    # file_name -> list of polygons
    file2polys = defaultdict(list)
    for a in ann["annotations"]:
        file2polys[id2file[a["image_id"]]].append((a["segmentation"], id2hw[a["image_id"]]))

    names = sorted(f for f in file2polys if os.path.exists(os.path.join(TRAIN_DIR, f)))
    n = len(names)
    print(f"[cache] building full-frame cache for {n} images "
          f"(~{2 * n * FULL * FULL / 1e9:.1f} GB, one-off)...")

    images = np.lib.format.open_memmap(img_path, mode="w+", dtype=np.uint8, shape=(n, FULL, FULL))
    masks = np.lib.format.open_memmap(msk_path, mode="w+", dtype=np.uint8, shape=(n, FULL, FULL))
    fg = np.zeros((n, FG_POINTS, 2), dtype=np.int16)
    cnt = np.zeros(n, dtype=np.int32)
    rng = np.random.RandomState(SEED)
    sl = slice(OFFSET, OFFSET + CROP)
    kept_px = dropped_px = 0

    t0 = time.time()
    for i, fn in enumerate(names):
        img = cv2.imread(os.path.join(TRAIN_DIR, fn), cv2.IMREAD_GRAYSCALE)
        if img.shape != (FULL, FULL):
            img = cv2.resize(img, (FULL, FULL), interpolation=cv2.INTER_AREA)
        images[i] = img

        merged = np.zeros((FULL, FULL), dtype=np.uint8)
        for seg, (ah, aw) in file2polys[fn]:
            rles = coco_mask.frPyObjects(seg, ah, aw)
            m = coco_mask.decode(coco_mask.merge(rles))
            if m.shape != (FULL, FULL):
                m = cv2.resize(m, (FULL, FULL), interpolation=cv2.INTER_NEAREST)
            np.maximum(merged, m, out=merged)
        masks[i] = merged

        # the crop is only safe if it throws away no ground truth -- keep score
        kept = int(merged[sl, sl].sum())
        kept_px += kept
        dropped_px += int(merged.sum()) - kept

        # anchor points come from inside the crop: tiles never start outside it
        ys, xs = np.nonzero(merged[sl, sl])
        ys, xs = ys + OFFSET, xs + OFFSET
        if len(ys):
            pick = rng.choice(len(ys), size=min(FG_POINTS, len(ys)), replace=False)
            fg[i, :len(pick)] = np.stack([ys[pick], xs[pick]], axis=1)
            cnt[i] = len(pick)

        if (i + 1) % 50 == 0 or i + 1 == n:
            print(f"  {i + 1}/{n}  ({time.time() - t0:.0f}s)")

    images.flush()
    masks.flush()
    np.save(fg_path, fg)
    np.save(cnt_path, cnt)
    json.dump(names, open(names_path, "w"))
    print(f"[cache] done in {time.time() - t0:.0f}s")
    print(f"[cache] filament pixels kept {kept_px:,} / dropped by the crop {dropped_px:,} "
          f"({100 * dropped_px / max(kept_px + dropped_px, 1):.4f}%)")
    return img_path, msk_path, fg_path, cnt_path, names

# --------------------------------------------------------------------------- #
# mosaic geometry
# --------------------------------------------------------------------------- #
def tile_starts():
    \"\"\"Start coordinate of each tile along one axis, in full-frame coords.

    TILES positions spread evenly from the crop's near edge to its far edge, so the
    first and last tiles sit flush against the crop and nothing is left uncovered.
    Rounding shares any leftover pixel out between the gaps rather than piling it all
    on the last one, so neighbouring overlaps differ by at most a pixel.
    \"\"\"
    span = CROP - TILE
    return [OFFSET + round(i * span / (TILES - 1)) for i in range(TILES)]


def tile_layout():
    \"\"\"One (tile_r, tile_c) per tile, full-frame coords. TILES^2 of them.\"\"\"
    starts = tile_starts()
    return [(r, c) for r in starts for c in starts]


def blend_window():
    \"\"\"TILE x TILE weight map: flat 1 through the middle, tapering to BLEND_FLOOR at
    the rim. This is what makes sense of a pixel that several tiles predicted.

    A pixel on a tile's edge was predicted with context on one side only. The same
    pixel sits well inside the neighbouring tile, with context all round it. So each
    prediction is weighted by its distance from its OWN tile's edge and the total is
    divided by the total weight: the tile that saw the pixel properly dominates, and
    the two crossfade instead of meeting at a cut. A hard cut would leave a seam right
    where connected components get labelled, and a filament crossing it could break in
    two -- exactly the one-to-many fragmentation the submission is scored on.

    The ramp is a raised cosine over OVERLAP px, the exact width two neighbours share,
    so one tile fades out as the other fades in and their weights sum to ~1 across the
    band. It stops at BLEND_FLOOR rather than 0 because the outermost ring of the crop
    is covered by ONE tile: there the floor is the only weight present, cancels in the
    division and leaves that tile's prediction untouched. A true zero would be 0/0.

    The window is separable, so the floor at a corner is BLEND_FLOOR^2 -- still a
    common factor between whatever tiles cover it, and still cancelling.
    \"\"\"
    d = np.minimum(np.arange(TILE), np.arange(TILE)[::-1]).astype(np.float32)
    ramp = 0.5 * (1.0 - np.cos(np.pi * np.clip(d / max(OVERLAP, 1), 0.0, 1.0)))
    line = BLEND_FLOOR + (1.0 - BLEND_FLOOR) * ramp
    return np.outer(line, line).astype(np.float32)


BLEND = blend_window()          # TILE^2 floats, built once and reused for every image
_BLEND_GPU = None               # its device copy, made on first use


def tile_at(arr, tile_r, tile_c):
    \"\"\"TILE x TILE slice of a full-frame array -- always fully inside the frame.\"\"\"
    return np.ascontiguousarray(arr[tile_r:tile_r + TILE, tile_c:tile_c + TILE])


def normalize(batch_u8, centre, scale):
    \"\"\"uint8 (B, H, W) numpy -> (B, 1, H, W) tensor on DEVICE, per-image normalized.\"\"\"
    x = torch.from_numpy(batch_u8).to(DEVICE).float().sub_(centre).div_(scale)
    return x.unsqueeze(1).to(memory_format=torch.channels_last)


@torch.no_grad()
def mosaic_predict(model, img):
    \"\"\"Reassemble a FULL x FULL probability map from OVERLAPPING tiles.

    img is a uint8 FULL x FULL array. Each tile's prediction is accumulated into the
    canvas under the blend window, and the window itself is accumulated alongside it;
    dividing the two at the end turns the sum into a weighted average, so a pixel that
    several tiles predicted gets a single reconciled probability rather than whichever
    tile happened to be written last.

    Pixels outside the centred crop are left at 0 -- no tile touches them, so both
    accumulators stay 0 there and the clamped division returns 0. They are black sky
    outside the solar disk and hold no ground truth.
    \"\"\"
    global _BLEND_GPU
    if _BLEND_GPU is None:
        _BLEND_GPU = torch.from_numpy(BLEND).to(DEVICE)
    win = _BLEND_GPU

    def forward(x):
        \"\"\"Mean sigmoid over the first TTA elements of the dihedral group.

        Training applies flips and RandomRotate90(p=1.0), so the model is already close
        to D4-equivariant and the transforms mostly disagree on noise. Averaging cancels
        that, which matters here specifically because the noise it cancels is what
        breaks a faint filament into two components and costs one-to-many.

        Each transform is undone on the OUTPUT before averaging, so this averages
        predictions in image space, never images.
        \"\"\"
        if TTA <= 1:
            return torch.sigmoid(model(x))
        outs = []
        for flip in range(2 if TTA >= 8 else 1):
            xf = torch.flip(x, dims=[-1]) if flip else x
            for k in range(4 if TTA >= 4 else 1):
                xr = torch.rot90(xf, k, dims=(-2, -1))
                y = torch.sigmoid(model(xr.contiguous(memory_format=torch.channels_last)))
                y = torch.rot90(y, -k, dims=(-2, -1))
                outs.append(torch.flip(y, dims=[-1]) if flip else y)
        return torch.stack(outs).mean(0)

    centre, scale = norm_stats(img)
    layout = tile_layout()
    acc = torch.zeros((FULL, FULL), device=DEVICE, dtype=torch.float32)
    wsum = torch.zeros((FULL, FULL), device=DEVICE, dtype=torch.float32)
    for k in range(0, len(layout), TILE_BATCH):
        chunk = layout[k:k + TILE_BATCH]
        tiles = np.stack([tile_at(img, tr, tc) for tr, tc in chunk])
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.device(DEVICE).type == "cuda"):
            probs = forward(normalize(tiles, centre, scale)).float()
        for (tr, tc), p in zip(chunk, probs):
            acc[tr:tr + TILE, tc:tc + TILE] += p[0] * win
            wsum[tr:tr + TILE, tc:tc + TILE] += win
    return acc / wsum.clamp(min=1e-6)

# --------------------------------------------------------------------------- #
# data
# --------------------------------------------------------------------------- #
# Augmentation strength lives here because the sampling margin is derived from it --
# keep them together so the two can never drift apart.
AUG_ROTATE = 25          # degrees
AUG_SCALE = (0.9, 1.1)
AUG_TRANSLATE = 0.05     # fraction of the window

TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=1.0),
    A.Affine(scale=AUG_SCALE, translate_percent=(-AUG_TRANSLATE, AUG_TRANSLATE),
             rotate=(-AUG_ROTATE, AUG_ROTATE), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
])

# A tile rotated by AUG_ROTATE and scaled down to AUG_SCALE[0] pulls content in from
# beyond its own edges. Augmenting the bare tile would fill that with black, putting
# invented pixels inside the tile the model is supposed to read as real sky. So crops
# are read WORK-wide, augmented, then cut back to TILE: the fill comes from genuine
# neighbouring pixels instead. Solving M >= TILE*(reach-1)/2 + AUG_TRANSLATE*(TILE+2M):
_reach = (math.cos(math.radians(AUG_ROTATE)) + math.sin(math.radians(AUG_ROTATE))) / AUG_SCALE[0]
AUG_MARGIN = int(math.ceil((TILE * (_reach - 1) / 2 + AUG_TRANSLATE * TILE)
                           / (1 - 2 * AUG_TRANSLATE)))
WORK = TILE + 2 * AUG_MARGIN


def work_window(arr, tile_r, tile_c):
    \"\"\"WORK x WORK window centred on the tile at (tile_r, tile_c), full-frame coords.

    Only tiles hard against the frame edge need padding, and there the image is black
    sky beyond the solar limb anyway, so a zero fill is what the sensor would show.
    \"\"\"
    r0, c0 = tile_r - AUG_MARGIN, tile_c - AUG_MARGIN
    r1, c1 = r0 + WORK, c0 + WORK
    pad_top, pad_left = max(0, -r0), max(0, -c0)
    pad_bot, pad_right = max(0, r1 - arr.shape[0]), max(0, c1 - arr.shape[1])
    sub = np.ascontiguousarray(arr[max(0, r0):min(arr.shape[0], r1),
                                   max(0, c0):min(arr.shape[1], c1)])
    if pad_top or pad_left or pad_bot or pad_right:
        sub = cv2.copyMakeBorder(sub, pad_top, pad_bot, pad_left, pad_right,
                                 cv2.BORDER_CONSTANT, value=0)
    return sub


class FilamentCrops(Dataset):
    \"\"\"Random TILE-sized crops from the full-res cache.

    FG_PROB of the crops are positioned so a known filament pixel lands somewhere
    inside the tile; the rest are uniform over the image, so the model still sees
    empty sky, the limb, and background artefacts.

    Each crop is read WORK-wide, augmented, then cut back to TILE, so the geometric
    augmentations pull in real neighbouring pixels rather than inventing black.
    \"\"\"

    def __init__(self, img_path, msk_path, fg_path, cnt_path, norm, indices):
        self.img_path, self.msk_path = img_path, msk_path
        self.fg = np.load(fg_path)
        self.cnt = np.load(cnt_path)
        self.norm = norm
        self.indices = indices
        self.images = None
        self.masks = None

    def __len__(self):
        return len(self.indices) * CROPS_PER_IMAGE

    def __getitem__(self, i):
        if self.images is None:  # open memmaps lazily (per worker)
            self.images = np.load(self.img_path, mmap_mode="r")
            self.masks = np.load(self.msk_path, mmap_mode="r")
        j = int(self.indices[i // CROPS_PER_IMAGE])

        # a tile origin is anywhere the tile still fits inside the crop; inference
        # uses a fixed overlapping grid, but training has no reason to be tied to it
        lo, hi = 0, CROP - TILE                        # valid tile origins, crop coords

        if random.random() < FG_PROB and self.cnt[j] > 0:
            y, x = self.fg[j, random.randrange(int(self.cnt[j]))]
            # offset randomly so the filament pixel can land anywhere in the tile
            r = int(np.clip(int(y) - OFFSET - random.randrange(TILE), lo, hi))
            c = int(np.clip(int(x) - OFFSET - random.randrange(TILE), lo, hi))
        else:
            r = random.randrange(lo, hi + 1)
            c = random.randrange(lo, hi + 1)

        tile_r, tile_c = OFFSET + r, OFFSET + c
        img = work_window(self.images[j], tile_r, tile_c)
        msk = work_window(self.masks[j], tile_r, tile_c)

        out = TRAIN_AUG(image=img, mask=msk)
        img, msk = out["image"], out["mask"]

        # cut the augmented window back to the tile the model sees. The window is
        # square and the tile is centred in it, so flips and 90-degree rotations map
        # that centre window onto itself.
        cut = slice(AUG_MARGIN, AUG_MARGIN + TILE)
        img, msk = img[cut, cut], msk[cut, cut]

        # the whole tile is supervised, so the label is all the model needs
        label = (np.ascontiguousarray(msk) > 0).astype(np.float32)
        # the frame's own centre/scale, not the tile's: a tile that happens to be all
        # quiet disk must not be stretched to look like one containing a filament
        centre, scale = self.norm[j]
        x_t = torch.from_numpy(np.ascontiguousarray(img)).float().sub_(centre).div_(scale)
        return x_t.unsqueeze(0), torch.from_numpy(label).unsqueeze(0)

# --------------------------------------------------------------------------- #
# model
# --------------------------------------------------------------------------- #
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Bottleneck(nn.Module):
    \"\"\"Residual squeeze-transform-expand block sitting between encoder and decoder.

    The encoder hands over its deepest stage (/32) and the decoder starts upsampling
    from it immediately, so nothing ever mixes information across that map at full
    width. This block does: 1x1 down to BOTTLENECK_CH, a stack of 3x3 convs with
    growing dilation (context out to several hundred input pixels at /32, which is the
    scale a filament spine spans), 1x1 back up, added residually.

    The final BatchNorm is zero-initialised, so at step 0 the block is exactly the
    identity and training starts from the same place the plain U-Net would.
    \"\"\"

    def __init__(self, ch, mid, dilations):
        super().__init__()
        self.reduce = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.convs = nn.Sequential(*[
            nn.Sequential(nn.Conv2d(mid, mid, 3, padding=d, dilation=d, bias=False),
                          nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
            for d in dilations])
        self.expand = nn.Sequential(
            nn.Conv2d(mid, ch, 1, bias=False), nn.BatchNorm2d(ch))
        nn.init.zeros_(self.expand[1].weight)   # start as identity
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.expand(self.convs(self.reduce(x))))


class UNet(nn.Module):
    \"\"\"timm features_only encoder + U-Net decoder that upsamples back to /1.

    The decoder is driven by the encoder's stage reductions, so it works whether
    the encoder exposes 5 stages starting at /2 (resnet34: 2 4 8 16 32) or 4
    stages starting at /4 (efficientvit_b1: 4 8 16 32). Levels without a matching
    encoder stage simply get no skip connection.
    \"\"\"

    def __init__(self, pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(
            ENCODER, pretrained=pretrained, features_only=True, in_chans=1
        )
        enc_ch = self.encoder.feature_info.channels()
        self.enc_red = self.encoder.feature_info.reduction()
        red2ch = dict(zip(self.enc_red, enc_ch))

        deepest = max(self.enc_red)
        n_up = int(math.log2(deepest))          # /32 -> 5 upsampling steps to /1
        cin = enc_ch[-1]
        blocks, skip_reds = [], []
        for i in range(n_up):
            red = deepest // (2 ** (i + 1))     # resolution level after this upsample
            skip_ch = red2ch.get(red, 0)
            cout = DECODER_CH[min(i, len(DECODER_CH) - 1)]
            blocks.append(ConvBlock(cin + skip_ch, cout))
            skip_reds.append(red if skip_ch else None)
            cin = cout
        self.blocks = nn.ModuleList(blocks)
        self.skip_reds = skip_reds
        self.bottleneck = (Bottleneck(enc_ch[-1], BOTTLENECK_CH, BOTTLENECK_DIL)
                           if BOTTLENECK_CH else nn.Identity())
        self.head = nn.Conv2d(cin, 1, 1)

    def forward(self, x):
        # The decoder doubles resolution 5x, so the encoder stages only line up with
        # their skips when the input is a multiple of 32. The mosaic geometry is solved
        # so TILE already is one (1056 = 33*32), which is the whole point of the
        # TILE % 32 assert up top -- this pad is a no-op at the configured geometry and
        # only fires if someone retunes TILE off the grid, in which case it adds <32px
        # of reflected border that the crop below takes straight back off.
        h, w = x.shape[-2:]
        pad_h, pad_w = (-h) % 32, (-w) % 32
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

        feats = self.encoder(x)
        red2feat = dict(zip(self.enc_red, feats))
        out = self.bottleneck(feats[-1])
        for block, red in zip(self.blocks, self.skip_reds):
            out = F.interpolate(out, scale_factor=2, mode="nearest")
            if red is not None:
                out = torch.cat([out, red2feat[red]], dim=1)
            out = block(out)
        out = self.head(out)
        return out[..., :h, :w] if (pad_h or pad_w) else out


def build_param_groups(model):
    \"\"\"One optimizer group per model part, so each part trains at its own LR.

    Within every part, params with ndim <= 1 (norm weights, biases) are split off
    into a weight_decay=0 group -- the usual pairing once groups are built by hand.
    \"\"\"
    parts = (("encoder", model.encoder, ENCODER_LR),
             ("bottleneck", model.bottleneck, BOTTLENECK_LR),
             ("decoder", model.blocks, DECODER_LR),
             ("head", model.head, HEAD_LR))

    groups, covered = [], 0
    for part, module, lr in parts:
        trainable = [p for p in module.parameters() if p.requires_grad]
        decay = [p for p in trainable if p.ndim > 1]
        no_decay = [p for p in trainable if p.ndim <= 1]
        if decay:
            groups.append({"params": decay, "lr": lr, "weight_decay": WEIGHT_DECAY, "part": part})
        if no_decay:
            groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0, "part": part})
        n_part = sum(p.numel() for p in trainable)
        covered += n_part
        print(f"[optim] {part:8s} lr={lr:.1e}  {n_part / 1e6:6.2f}M params  "
              f"({len(decay)} decayed / {len(no_decay)} not)")

    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert covered == total, f"param groups cover {covered} of {total} trainable params"
    return groups

# --------------------------------------------------------------------------- #
# loss / metric
# --------------------------------------------------------------------------- #
def criterion(logits, target, pos_weight):
    \"\"\"BCE + soft Dice over the whole tile.

    Every pixel of the tile is supervised: overlapping tiles are reconciled at
    inference by the blend window, not by masking part of the loss away, so there is
    no region to weight down and the two terms are taken flat.
    \"\"\"
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)

    # .float() is load-bearing under float16 autocast: p.sum(1) runs over TILE^2 =
    # 1,115,136 probabilities, and float16 tops out at 65,504. A tile more than ~6%
    # foreground would overflow the sum to inf and take the whole loss with it.
    prob = torch.sigmoid(logits.float())
    p, t = prob.flatten(1), target.flatten(1)
    # per SAMPLE, then averaged -- not pooled over the batch. Pooling would let one
    # tile packed with filament carry the gradient for a tile that has almost none.
    inter = (p * t).sum(1)
    dice = (1.0 - (2.0 * inter + DICE_EPS)
            / (p.sum(1) + t.sum(1) + DICE_EPS)).mean()

    return BCE_W * bce + DICE_W * dice

# --------------------------------------------------------------------------- #
# instances + Panoptic Quality (the ranking metric)
# --------------------------------------------------------------------------- #""",
    'diagnostics.py': """\"\"\"Compact image-space audit with component and instance overlays.\"\"\"
import cv2
import numpy as np

from filament_seg.rle import rle_to_mask


def montage(path, gray, pg, pb, fused, components, raw, final, links):
    rng = np.random.default_rng(3)

    def heat(p):
        return cv2.applyColorMap(np.uint8(np.clip(p, 0, 1) * 255), cv2.COLORMAP_TURBO)

    def overlay(instances, annotate=False):
        panel = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        linked = {row["key"]: len(row["linked"]) for row in links}
        for inst in instances:
            mask = rle_to_mask(inst["rle"]).astype(bool)
            panel[mask] = rng.integers(65, 255, size=3, dtype=np.uint8)
            if annotate and inst["key"] in linked:
                x, y = map(int, inst["box"][:2])
                cv2.putText(panel, str(linked[inst["key"]]), (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                            1.4, (255, 255, 255), 2)
        return panel

    colors = rng.integers(65, 255, size=(int(components.max()) + 1, 3), dtype=np.uint8)
    colors[0] = 0
    views = [("Observation", cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)),
             ("GitHub probability", heat(pg)), ("Blended probability", heat(pb)),
             ("Feather probability", heat(fused)), ("High foreground components", colors[components]),
             ("YOLO / linked region counts", overlay(raw, True)), ("Final instances", overlay(final)),
             ("Recovered instances", overlay([i for i in final if i["source"] == "semantic"]))]
    panels = []
    for title, img in views:
        panel = cv2.resize(img, (480, 480), interpolation=cv2.INTER_AREA)
        cv2.rectangle(panel, (0, 0), (479, 25), (0, 0, 0), -1)
        cv2.putText(panel, title, (8, 18), cv2.FONT_HERSHEY_SIMPLEX, .52, (255, 255, 255), 1)
        panels.append(panel)
    if not cv2.imwrite(str(path), np.vstack([np.hstack(panels[:4]), np.hstack(panels[4:])])):
        raise IOError(f"Failed to save {path}")
""",
    'driver.py': """\"\"\"End-to-end Kaggle entry point. The outer fold is never used for training or tuning.\"\"\"
from dataclasses import asdict
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import shutil
import time

import numpy as np
import pandas as pd
from scipy.special import expit

from filament_seg.data import load_annotations, make_split, stems_of, test_image_paths
from filament_seg.fusion import FusionParams, fuse, merge_views
from filament_seg.rle import build_submission, labels_to_rles
from official_pq import require_disjoint
from recovery import Rules, blend, components, finalize, orphans, supported_candidates
from evaluate import score, summarize
from submission_audit import audit_csv


WEIGHTS = {
    "validated_unet_run3_last.pt": "1d1dfd8042b44a198d52efb4c5b928abbf861cbe0e973a1ee77174ad19bf06f9",
    "validated_det1_yolo11s_1024_best.pt": "d61d42124926766d7e08f5da010adeed2d2eadb58de0c7538e87ba5857d358ef",
    "validated_det2_yolo11m_1280_best.pt": "bbfd9ffbc44d8d6415078455765db286af00dec61dc7449909d22fdd9a4773ca",
    "validated_det4_yolo11m_1536_last.pt": "d70eea1d195db1ea11055bfad906e8c4bc57fec16cd85c8248cfd737cb80af25",
}
ANN = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
CENTER = (88, 1960)
RULES = Rules()
GH_RULES = FusionParams(threshold=.5, close_radius=3, grow=4, min_score=.30, min_area=200,
                       keep_unclaimed=0, unclaimed_gap=24)


def dump(path, payload):
    def convert(value):
        if isinstance(value, bytes):
            return value.decode("ascii")
        raise TypeError(type(value).__name__)

    path.write_text(json.dumps(payload, default=convert, ensure_ascii=False), encoding="utf-8")


def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def find(root, name, checksum=None):
    paths = sorted(root.rglob(name))
    if checksum:
        paths = [p for p in paths if digest(p) == checksum]
    if not paths:
        raise FileNotFoundError(f"Missing or checksum mismatch: {name} under {root}")
    if checksum is None and len(paths) != 1:
        raise ValueError(f"Ambiguous input for {name}: {paths}")
    return paths[0]


def preflight(root):
    ann = find(root, ANN)
    weights = [find(root, name, checksum) for name, checksum in WEIGHTS.items()]
    data = load_annotations(ann)
    split = make_split(data, group_by="date", seed=0, val_fraction=.2)
    training, validation = (stems_of(data, split[k]) for k in ("train", "val"))
    train_dates, val_dates = ({data.images[i].date for i in split[k]} for k in ("train", "val"))
    tests = test_image_paths(ann.parent.parent / "test" / "test_images")
    if (len(training), len(validation), len(tests)) != (563, 144, 180):
        raise ValueError(f"Unexpected split size: {len(training)}, {len(validation)}, {len(tests)}")
    grouped = data.by_file_stem()
    if set(training) & set(validation) or train_dates & val_dates:
        raise ValueError("Training and validation overlap")
    for key, stems in (("train", training), ("val", validation)):
        if set(split[key]) != {i for stem in stems for i in grouped[stem]}:
            raise ValueError(f"{key} omits annotator records")
    train_dir = ann.parent / "train_images"
    for record in data.images.values():
        if not (train_dir / record.file_name).is_file():
            raise FileNotFoundError(train_dir / record.file_name)
    if len({p.stem for p in tests}) != len(tests) or set(grouped) & {p.stem for p in tests}:
        raise ValueError("Invalid test image IDs")
    print(f"Verified GitHub weights and 563/144/180 split at {ann.parent.parent}", flush=True)
    return ann, weights, data, split, validation, tests


def semantic_only(prob, valid):
    labels, stats = components((prob >= RULES.high) & valid)
    labels[~(stats[:, 4] >= RULES.min_area)[labels]] = 0
    return labels_to_rles(labels)


def run_image(image, models, artifacts, visualize, export_maps, evaluate_all):
    start = time.perf_counter()
    gray, disk, logits, pb, raw, views = models.predict(image)
    model_seconds = time.perf_counter() - start
    pg = expit(logits)
    valid = np.zeros(disk.shape, dtype=bool)
    a, b = CENTER
    valid[a:b, a:b] = True
    if np.any(pb[~valid] != 0) or not np.any(pb[valid] > 0):
        raise ValueError(f"Unexpected Blended projection on {image.name}")
    fused = blend(pg, pb, CENTER, RULES.weight, RULES.feather)
    variants, recovered = {}, []
    if evaluate_all:
        baseline = labels_to_rles(fuse(logits, disk, merge_views(views), GH_RULES))
        blended = semantic_only(pb, disk & valid)
        variants.update(github=(baseline, ["github"] * len(baseline)),
                        blended=(blended, ["blended"] * len(blended)))
        probabilities = dict(pg=pg, mean=blend(pg, pb, CENTER, .5, 0),
                             weighted=blend(pg, pb, CENTER, RULES.weight, 0), feather=fused)
    else:
        probabilities = dict(feather=fused)

    for key, prob in probabilities.items():
        candidates, labels, links = supported_candidates(raw, prob, disk, RULES)
        kept, claimed, decisions = finalize(candidates, disk.shape, RULES)
        variants[key] = ([x["rle"] for x in kept], [x["source"] for x in kept])
        if key != "feather":
            continue
        recovered, grades = orphans(labels, claimed, pg, pb, prob, valid, gray, RULES)
        # Ownership of the YOLO-only ablation is frozen when recovery is enabled.
        final, _, final_decisions = finalize(kept + recovered, disk.shape, RULES)
        variants["recovery"] = ([x["rle"] for x in final], [x["source"] for x in final])
        require_disjoint(variants["recovery"][0], disk.shape)
        dump(artifacts / f"{image.stem}.json",
             dict(raw=raw, guided=candidates, graph=links, orphan_grades=grades,
                  decisions=decisions, final_decisions=final_decisions, final=final,
                  counts=dict(raw=len(raw), accepted=len(candidates), assessed_orphans=len(grades),
                              recovered=len(recovered), final=len(final),
                              suppressed=len(candidates) + len(recovered) - len(final)),
                  model_seconds=model_seconds, seconds=time.perf_counter() - start))
        if visualize:
            from diagnostics import montage
            montage(artifacts / f"{image.stem}.jpg", gray, pg, pb, fused, labels, raw, final, links)
        if export_maps or visualize:
            if shutil.disk_usage(artifacts).free < 2 * 1024**3:
                raise RuntimeError("Less than 2 GiB free for probability archives")
            np.savez_compressed(artifacts / f"{image.stem}.npz", pg=pg.astype(np.float16),
                                pb=pb.astype(np.float16), fused=fused.astype(np.float16),
                                valid=valid)
    return variants, [x["rle"] for x in raw], [x["rle"] for x in recovered]


def run(config):
    root = Path(config["input_root"])
    work = Path(config["working_root"])
    out = work / "solar_v3"
    out.mkdir(parents=True, exist_ok=True)
    ann, weights, data, split, validation, tests = preflight(root)
    dump(out / "split.json", split)
    if config["mode"] == "preflight":
        return
    if not config["accept_local_blended_source"] and not config["exact_v10_source"]:
        raise RuntimeError("V10 source identity is unverified. Supply exact V10 when building, "
                           "or explicitly accept the local Blended architecture in the cell settings.")

    if not config["exact_v10_source"]:
        print("Blended source: supplied local architecture (exact V10 identity unverified)", flush=True)

    import torch
    import blended_source as source
    from train_blended import fit, restore

    if not torch.cuda.is_available():
        raise RuntimeError("Kaggle GPU is required for the full V3 model path")
    if (source.FULL, source.CROP, source.OFFSET) != (2048, 1872, 88):
        raise RuntimeError("Blended source geometry differs from V3 contract")
    source.TILE_BATCH = config["mosaic_batch"]
    source.BATCH_SIZE = config["train_batch"]
    signature = dict(annotation_sha256=digest(ann), source_sha256=config["source_sha256"],
                     runtime_source_sha256=digest(Path(source.__file__)),
                     train_ids=split["train"], train_batch=config["train_batch"],
                     seed=source.SEED, epochs=source.EPOCHS)
    packages = {name: version(name) for name in (
        "torch", "torchvision", "timm", "albumentations", "ultralytics",
        "segmentation-models-pytorch", "pycocotools", "numpy", "scipy", "tqdm",
    )}
    code = {p.name: digest(p) for p in Path(__file__).parent.glob("*.py")}
    dump(out / "manifest.json", dict(config=config, weights=dict(zip(map(str, weights), WEIGHTS.values())),
                                     recipe=signature, rules=asdict(RULES), github=asdict(GH_RULES),
                                     dependencies=packages, code_sha256=code))
    checkpoint = out / "blended_shared_split_last.pt"
    configured = config["blended_checkpoint"]
    mounted = sorted(root.rglob(checkpoint.name))
    if configured:
        reusable = Path(configured)
    elif checkpoint.is_file():
        reusable = checkpoint
    elif len(mounted) == 1:
        reusable = mounted[0]
    elif mounted:
        raise ValueError("Several Blended checkpoints found. Set blended_checkpoint explicitly.")
    else:
        reusable = None
    if reusable is None:
        if config["mode"] in ("submit", "smoke"):
            raise FileNotFoundError("This mode requires a trained Blended checkpoint")
        scratch = Path(config["scratch_root"]) / signature["annotation_sha256"][:16]
        model = fit(source, ann, ann.parent / "train_images", split["train"],
                    scratch, checkpoint, signature, config["train_batch"])
    else:
        model = restore(source, reusable, signature)
    selected = reusable or checkpoint
    dump(out / "blended_checkpoint.json", dict(path=str(selected), sha256=digest(selected)))
    print(f"Blended checkpoint ready: {selected}", flush=True)
    if config["mode"] == "train":
        return

    from predict import Models

    models = Models(weights, (1024, 1280, 1536), source, model)
    val_artifacts = out / "validation"
    test_artifacts = out / "test"
    for directory in (val_artifacts, test_artifacts):
        directory.mkdir(exist_ok=True)
    grouped = data.by_file_stem()
    rows = []
    mode = config["mode"]
    if mode in ("full", "smoke", "eval"):
        for index, stem in enumerate(validation[:3] if mode == "smoke" else validation):
            ids = grouped[stem]
            path = ann.parent / "train_images" / data.images[ids[0]].file_name
            variants, raw, recovered = run_image(path, models, val_artifacts, index < 3,
                                                 config["export_all_maps"], True)
            scored = score(stem, data.gt_dict(ids), variants, raw, recovered)
            for row in scored:
                row.update(date=data.images[ids[0]].date, observatory=data.images[ids[0]].observatory)
            rows.extend(scored)
            print(f"validation {index + 1}: {stem}", flush=True)
        table = pd.DataFrame(rows)
        table.to_csv(out / "validation_records.csv", index=False)
        result = pd.DataFrame(summarize(table))
        result.to_csv(out / "ablation_pq.csv", index=False)
        print(result.to_string(index=False))
    if mode in ("full", "submit", "smoke"):
        predictions = {}
        for index, path in enumerate(tests[:3] if mode == "smoke" else tests):
            variants, _, _ = run_image(path, models, test_artifacts, index < 3,
                                       config["export_all_maps"], False)
            predictions[path.stem] = variants["recovery"][0]
            print(f"test {index + 1}: {path.stem}", flush=True)
        if mode == "smoke":
            return
        coverage = dict(all_test_photos_processed=set(predictions) == {p.stem for p in tests},
                        counts_by_photo={stem: len(rles) for stem, rles in predictions.items()})
        temporary = out / "candidate_submission.csv"
        build_submission(predictions).to_csv(temporary, index=False)
        report = audit_csv(temporary, [p.stem for p in tests], coverage)
        dump(out / "submission_audit.json", {**report, **coverage})
        temporary.replace(work / "submission.csv")
        print(f"Validated submission: {work / 'submission.csv'}", flush=True)
""",
    'evaluate.py': """\"\"\"Pooled official scores and per-annotator candidate attribution.\"\"\"
import numpy as np
from pycocotools import mask as cm

from official_pq import Totals, from_iou, pairwise_iou


def score(stem, records, variants, raw, recovered):
    rows = []
    for record, gt in records.items():
        areas = np.array([cm.area(r) for r in gt])
        raw_hit = (pairwise_iou(gt, raw) > .5).any(axis=1)
        augmented_hit = (pairwise_iou(gt, raw + recovered) > .5).any(axis=1)
        no_recovery = (pairwise_iou(gt, variants["feather"][0]) > .5).any(axis=1)
        for name, (pred, sources) in variants.items():
            iou = pairwise_iou(gt, pred)
            hit = iou > .5
            recall = hit.any(axis=1)
            semantic = np.array([s == "semantic" for s in sources], dtype=bool)
            row = dict(stem=stem, record=record, experiment=name, **from_iou(iou).summary())
            row.update(raw_oracle=int(raw_hit.sum()), augmented_oracle=int(augmented_hit.sum()),
                       semantic_tp=int(hit[:, semantic].sum()),
                       semantic_fp=int((~hit[:, semantic].any(axis=0)).sum()),
                       gained_gt=int((recall & ~no_recovery).sum()),
                       lost_gt=int((~recall & no_recovery).sum()))
            for key, group in dict(under250=areas < 250, mid=(areas >= 250) & (areas < 1000),
                                   large=areas >= 1000).items():
                row[f"{key}_gt"] = int(group.sum())
                row[f"{key}_recalled"] = int((recall & group).sum())
            rows.append(row)
    return rows


def summarize(frame):
    rows = []
    for name, group in frame.groupby("experiment", sort=False):
        totals = Totals(**{key: group[key].sum().item() for key in Totals.__dataclass_fields__})
        row = dict(experiment=name, **totals.summary())
        for key in ("raw_oracle", "augmented_oracle", "semantic_tp", "semantic_fp", "gained_gt", "lost_gt"):
            row[key] = int(group[key].sum())
        for size in ("under250", "mid", "large"):
            gt = int(group[f"{size}_gt"].sum())
            hits = int(group[f"{size}_recalled"].sum())
            row.update({f"{size}_gt": gt, f"{size}_recalled": hits,
                        f"{size}_recall": hits / gt if gt else None})
        rows.append(row)
    return rows
""",
    'filament_seg/LICENSE': """MIT License

Copyright (c) 2026 Amey Pawar

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

""",
    'filament_seg/__init__.py': """\"\"\"Solar Filament Segmentation Challenge 2026 (IEEE Big Data Cup).

Package layout:

* :mod:`filament_seg.config`      paths and dataset constants
* :mod:`filament_seg.rle`         mask <-> COCO RLE <-> submission CSV
* :mod:`filament_seg.metrics`     local Panoptic Quality + rubric diagnostics
* :mod:`filament_seg.data`        MAGFiLO annotations and leak-free splits
* :mod:`filament_seg.disk`        solar disk detection, limb-darkening removal
* :mod:`filament_seg.postprocess` binary mask -> filament instances
* :mod:`filament_seg.baseline`    model-free detector (pipeline validation)
\"\"\"

__version__ = "0.1.0"

""",
    'filament_seg/baseline.py': """\"\"\"A model-free filament detector.

Purpose: get a *valid submission on the leaderboard before any training starts*,
so the RLE encoding, the CSV format and the local evaluator are all proven
against the real scorer. It is also the honest floor that any learned model has
to beat.

Method -- essentially the classical H-alpha filament recipe:

1. locate the solar disk and mask off everything outside it
2. divide out limb darkening so one threshold works at any radius
3. call a pixel a filament when it is ``k`` standard deviations darker than the
   quiet Sun around it
4. morphological cleanup, gap-bridging and an area filter to form instances

Expect a modest PQ. The value is in the plumbing, not the score.
\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np

from .disk import Disk, detect_disk, flatten, read_image
from .postprocess import PostprocessParams, binary_to_instances
from .rle import Rle, labels_to_rles


@dataclass
class BaselineParams:
    \"\"\"Thresholding knobs. Tune ``k`` first -- it dominates the score.\"\"\"

    #: Darkness threshold in standard deviations below the quiet-Sun level.
    #: 2.0 comes from a grid sweep over 40 validation images; PQ is fairly
    #: flat between k=1.6 and k=2.4 but collapses above k=2.8.
    k: float = 2.0
    #: Shrink factor applied to the disk radius; the limb itself is noisy.
    disk_shrink: float = 0.96
    #: Gaussian smoothing (in pixels) applied before thresholding.
    blur_sigma: float = 2.0
    postprocess: PostprocessParams = field(default_factory=PostprocessParams)


def segment(
    image: np.ndarray, params: BaselineParams | None = None
) -> tuple[np.ndarray, Disk]:
    \"\"\"Return an integer instance-label map and the detected disk.\"\"\"
    params = params or BaselineParams()

    disk = detect_disk(image)
    disk_mask = disk.mask(image.shape, shrink=params.disk_shrink)

    flat = flatten(image, disk)
    if params.blur_sigma > 0:
        flat = cv2.GaussianBlur(flat, (0, 0), sigmaX=params.blur_sigma)

    inside = flat[disk_mask]
    if inside.size == 0:
        return np.zeros(image.shape, dtype=np.int32), disk

    # Robust centre/spread: filaments are a small, dark minority of disk pixels,
    # so the median and the MAD are barely moved by them.
    median = float(np.median(inside))
    mad = float(np.median(np.abs(inside - median)))
    sigma = 1.4826 * mad if mad > 0 else float(inside.std())
    threshold = median - params.k * max(sigma, 1e-6)

    binary = (flat < threshold) & disk_mask
    labels = binary_to_instances(binary, params.postprocess, restrict_to=disk_mask)
    return labels, disk


def predict_rles(
    image_path: str | Path, params: BaselineParams | None = None
) -> list[Rle]:
    \"\"\"Predict filament instances for one image file, as RLEs.\"\"\"
    labels, _ = segment(read_image(image_path), params)
    return labels_to_rles(labels)

""",
    'filament_seg/config.py': """\"\"\"Project-wide paths and constants.

Every path can be overridden with an environment variable so the same code runs
unchanged locally and inside a Kaggle notebook (where the data lives under
``/kaggle/input``).
\"\"\"

from __future__ import annotations

import os
from pathlib import Path

# --- Layout -----------------------------------------------------------------

REPO_ROOT = Path(__file__).resolve().parent.parent

DATA_ROOT = Path(
    os.environ.get("FILAMENT_DATA_ROOT", REPO_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026")
)
OUTPUT_ROOT = Path(os.environ.get("FILAMENT_OUTPUT_ROOT", REPO_ROOT / "outputs"))

TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TRAIN_ANNOTATIONS = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"

SPLIT_PATH = OUTPUT_ROOT / "splits.json"

#: Preprocessing cache (flattened images, masks, disk geometry) that
#: scripts/preprocess.py builds and every other script reads.
CACHE_DIR = REPO_ROOT / "data" / "cache"

# --- Dataset constants ------------------------------------------------------

#: All GONG H-alpha observations in this competition are 2048x2048 grayscale.
#: The submission format relies on this being fixed (RLE size is not submitted).
IMAGE_HEIGHT = 2048
IMAGE_WIDTH = 2048
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)

#: Chirality classes exist in MAGFiLO but are irrelevant to this competition,
#: which is scored on masks only.
CATEGORY_NAMES = {1: "Left", 2: "Right", 3: "Unidentifiable", 4: "Ambiguous"}

# --- Evaluation constants ---------------------------------------------------

#: Panoptic Quality matches a prediction to a ground-truth segment when their
#: IoU exceeds this value. Above 0.5 the matching is guaranteed unique.
PQ_IOU_THRESHOLD = 0.5

#: Minimum overlap fraction used when counting one-to-many / many-to-one
#: relations for the fragmentation diagnostics (part of the judging rubric).
FRAGMENT_OVERLAP_THRESHOLD = 0.10


def ensure_output_dir() -> Path:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return OUTPUT_ROOT

""",
    'filament_seg/data.py': """\"\"\"Loading MAGFiLO annotations and building leak-free train/validation splits.

The dataset has one structural trap that quietly inflates validation scores:

* **The same observation is annotated by several annotators independently.** They
  appear as distinct ``image["id"]`` values that differ only in the batch prefix
  (``010101-2016...`` vs ``010102-2016...``) but share a ``file_name``. Splitting
  on ``image_id`` puts the same pixels in both train and validation.

* **Filaments live for days to weeks.** Two observations from the same day (or
  the same week) are near-duplicates. Splitting randomly by file name still
  leaks. ``group_by="date"`` is the default here; ``"month"`` is the stricter
  option and gives a more honest estimate of generalisation at the cost of a
  pessimistic bias relative to the leaderboard.
\"\"\"

from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Literal, Sequence

from .config import IMAGE_HEIGHT, IMAGE_WIDTH, TRAIN_ANNOTATIONS
from .rle import Rle, polygon_to_rle

GroupBy = Literal["file", "date", "month"]


@dataclass(frozen=True)
class ImageRecord:
    \"\"\"One annotated view of one observation.\"\"\"

    image_id: str  # e.g. "010401-20160920230134Lh"
    file_name: str  # e.g. "20160920230134Lh.jpg"
    width: int
    height: int

    @property
    def stem(self) -> str:
        \"\"\"Observation name without extension, e.g. ``20160920230134Lh``.\"\"\"
        return Path(self.file_name).stem

    @property
    def annotator(self) -> str:
        \"\"\"Annotator batch prefix, or ``""`` when the id carries none.\"\"\"
        return self.image_id.split("-", 1)[0] if "-" in self.image_id else ""

    @property
    def date(self) -> str:
        return self.stem[:8]

    @property
    def month(self) -> str:
        return self.stem[:6]

    @property
    def observatory(self) -> str:
        \"\"\"Two-letter GONG site code, e.g. ``Bh`` (Big Bear), ``Lh`` (Learmonth).\"\"\"
        return self.stem[14:16]

    def group_key(self, group_by: GroupBy = "date") -> str:
        return {"file": self.stem, "date": self.date, "month": self.month}[group_by]


@dataclass
class Annotations:
    \"\"\"Parsed COCO file, indexed the ways this project actually needs.\"\"\"

    images: dict[str, ImageRecord]
    annotations_by_image: dict[str, list[dict]]
    categories: dict[int, str]

    def __len__(self) -> int:
        return len(self.images)

    @property
    def image_ids(self) -> list[str]:
        return sorted(self.images)

    def by_file_stem(self) -> dict[str, list[str]]:
        \"\"\"``{observation stem: [image_id per annotator]}``.\"\"\"
        grouped: dict[str, list[str]] = {}
        for image_id, record in self.images.items():
            grouped.setdefault(record.stem, []).append(image_id)
        return {k: sorted(v) for k, v in sorted(grouped.items())}

    def gt_rles(self, image_id: str) -> list[Rle]:
        \"\"\"Ground-truth instance RLEs for one annotated image.\"\"\"
        record = self.images[image_id]
        out: list[Rle] = []
        for ann in self.annotations_by_image.get(image_id, []):
            segmentation = ann.get("segmentation")
            if not segmentation:
                continue
            out.append(polygon_to_rle(segmentation, record.height, record.width))
        return out

    def gt_dict(self, image_ids: Iterable[str]) -> dict[str, list[Rle]]:
        return {image_id: self.gt_rles(image_id) for image_id in image_ids}


def load_annotations(path: str | Path = TRAIN_ANNOTATIONS) -> Annotations:
    with open(path, "r", encoding="utf-8") as handle:
        raw = json.load(handle)

    images = {}
    for entry in raw.get("images", []):
        images[str(entry["id"])] = ImageRecord(
            image_id=str(entry["id"]),
            file_name=entry["file_name"],
            width=int(entry.get("width") or IMAGE_WIDTH),
            height=int(entry.get("height") or IMAGE_HEIGHT),
        )

    annotations_by_image: dict[str, list[dict]] = {}
    for ann in raw.get("annotations", []):
        annotations_by_image.setdefault(str(ann["image_id"]), []).append(ann)

    categories = {int(c["id"]): c["name"] for c in raw.get("categories", [])}
    return Annotations(images, annotations_by_image, categories)


def make_split(
    annotations: Annotations,
    val_fraction: float = 0.2,
    group_by: GroupBy = "date",
    seed: int = 0,
) -> dict[str, list[str]]:
    \"\"\"Split image ids into train/val so that no group spans both sides.

    Returns ``{"train": [...], "val": [...], "group_by": ...}`` with image ids
    (annotator-prefixed), which is what the loaders and the evaluator key on.
    \"\"\"
    groups: dict[str, list[str]] = {}
    for image_id, record in annotations.images.items():
        groups.setdefault(record.group_key(group_by), []).append(image_id)

    keys = sorted(groups)
    rng = random.Random(seed)
    rng.shuffle(keys)

    n_val = max(1, round(len(keys) * val_fraction))
    val_keys = set(keys[:n_val])

    train_ids = sorted(i for k in keys if k not in val_keys for i in groups[k])
    val_ids = sorted(i for k in keys if k in val_keys for i in groups[k])
    return {
        "group_by": group_by,
        "seed": seed,
        "val_fraction": val_fraction,
        "n_groups": len(keys),
        "train": train_ids,
        "val": val_ids,
    }


def deduplicate_by_file(
    annotations: Annotations, image_ids: Sequence[str], seed: int = 0
) -> list[str]:
    \"\"\"Keep one annotator per observation.

    Used for training, where the duplicates are label noise rather than extra
    signal. Validation should not use it: the official scorer matches the
    predictions against every annotator's view separately and pools the
    counts, so scoring against one random annotator optimises a different
    objective. See :func:`records_for_stems`.
    \"\"\"
    rng = random.Random(seed)
    chosen: list[str] = []
    grouped: dict[str, list[str]] = {}
    for image_id in image_ids:
        grouped.setdefault(annotations.images[image_id].stem, []).append(image_id)
    for stem in sorted(grouped):
        chosen.append(rng.choice(sorted(grouped[stem])))
    return sorted(chosen)


def load_split(path: str | Path) -> dict:
    \"\"\"The train/val split written by scripts/make_splits.py.\"\"\"
    return json.loads(Path(path).read_text(encoding="utf-8"))


def stems_of(annotations: Annotations, image_ids: Iterable[str]) -> list[str]:
    \"\"\"The observations (file stems) that ``image_ids`` are views of, sorted.\"\"\"
    return sorted({annotations.images[i].stem for i in image_ids})


def sample_stems(
    annotations: Annotations, image_ids: Iterable[str], n: int | None = None, seed: int = 0
) -> list[str]:
    \"\"\"A fixed random sample of ``n`` observations (all of them if ``n`` is falsy).

    Sampling observations rather than taking the first ``n`` sorted view ids
    matters: view ids start with the annotator batch, so a sorted prefix comes
    entirely from one batch of annotators. Shuffling with a fixed seed gives
    the same sample on every call, so per-epoch scores stay comparable.
    \"\"\"
    stems = stems_of(annotations, image_ids)
    random.Random(seed).shuffle(stems)
    return sorted(stems[:n] if n else stems)


def records_for_stems(
    annotations: Annotations, image_ids: Iterable[str], stems: Iterable[str]
) -> list[str]:
    \"\"\"Every annotator view among ``image_ids`` of the given observations.

    This is the unit the official scorer works in: one prediction set per
    observation, matched against each annotator's view of it separately, with
    TP/FP/FN pooled over all views.
    \"\"\"
    wanted = set(stems)
    return sorted(i for i in image_ids if annotations.images[i].stem in wanted)


def test_image_paths(directory: str | Path) -> list[Path]:
    directory = Path(directory)
    patterns = ("*.jpeg", "*.jpg", "*.JPEG", "*.JPG", "*.png")
    paths: list[Path] = []
    for pattern in patterns:
        paths.extend(directory.glob(pattern))
    return sorted(set(paths))

""",
    'filament_seg/dataset.py': """\"\"\"Cropped-tile dataset for training the filament segmentation network.

Two things shape this dataset more than anything else:

* **Filaments are rare.** They cover roughly 0.2% of disk pixels (median area
  1228 px in a 2048x2048 frame), so a uniformly random crop is empty the
  overwhelming majority of the time. ``filament_bias`` fixes this by centring
  most crops on an actual ground-truth filament, jittered so the network
  still has to localise rather than just find the middle of the crop.

* **Native resolution only.** Filaments are a few pixels wide; anything that
  resamples the image -- including the elastic/scale augmentation that is
  standard for most segmentation tasks -- destroys the structures the metric
  is graded on. Only flips, 90-degree rotations and brightness/contrast
  jitter are used here.

Everything expensive and parameter-independent (disk detection, limb
darkening) is expected to already live in ``scripts/preprocess.py``'s cache;
this module only reads that cache and crops it.
\"\"\"

from __future__ import annotations

import json
import random
from collections import OrderedDict
from pathlib import Path
from typing import Sequence

import albumentations as A
import cv2
import numpy as np
import torch
from pycocotools import mask as mask_utils
from torch.utils.data import Dataset

from .data import Annotations, deduplicate_by_file
from .disk import Disk

#: Flips and 90-degree rotations preserve every pixel exactly (no
#: interpolation), which is why they're safe for structures a few pixels
#: wide. Brightness/contrast jitter only ever touches the "image" target
#: below, never the "radius" one -- see the ``additional_targets`` note.
_AUGMENT = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    ],
    # albumentations only applies colour transforms to "image"-typed targets;
    # tagging the radius map as "mask" gets it the flip/rotation but keeps it
    # immune to brightness/contrast, which would corrupt its meaning (it's a
    # normalised coordinate, not an intensity).
    additional_targets={"radius": "mask"},
)


def load_disk_geometry(path: str | Path) -> dict[str, Disk]:
    \"\"\"Load ``scripts/preprocess.py``'s ``disk.json`` as ``{stem: Disk}``.\"\"\"
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    return {stem: Disk(**geometry) for stem, geometry in raw.items()}


def load_model_input(flat_dir: str | Path, stem: str, disk: Disk) -> np.ndarray | None:
    \"\"\"The 2-channel full-frame network input for one observation.

    Channel 0 is the cached limb-darkening-flattened intensity in [0, 1],
    channel 1 the normalised radius map. Returns None if the flat image is not
    cached, so callers can decide whether that is fatal.
    \"\"\"
    flat_u8 = cv2.imread(str(Path(flat_dir) / f"{stem}.png"), cv2.IMREAD_GRAYSCALE)
    if flat_u8 is None:
        return None
    radius = disk.radius_map(flat_u8.shape).astype(np.float32)
    return np.stack([flat_u8.astype(np.float32) / 255.0, radius], axis=0)


def _instance_centroids(annotations: Annotations, image_id: str) -> list[tuple[float, float]]:
    \"\"\"Bounding-box centre of each ground-truth filament in ``image_id``.

    A bbox centre is not the exact centroid of a curved ribbon, but it is a
    fine jitter target and, unlike a full decode, stays cheap even for the
    handful of images with two dozen instances.
    \"\"\"
    centroids = []
    for rle in annotations.gt_rles(image_id):
        x, y, w, h = mask_utils.toBbox(rle)
        centroids.append((float(y + h / 2.0), float(x + w / 2.0)))
    return centroids


class _BoundedCache:
    \"\"\"Tiny LRU cache of decoded flat images (4 MB each), bounded so memory
    stays flat over an epoch when the images are not preloaded. The sampler
    shuffles crops, so it rarely hits; decoding the PNG is the cost it saves.
    \"\"\"

    def __init__(self, maxsize: int = 8) -> None:
        self._maxsize = maxsize
        self._data: "OrderedDict[str, np.ndarray]" = OrderedDict()

    def get(self, key: str) -> np.ndarray | None:
        entry = self._data.get(key)
        if entry is not None:
            self._data.move_to_end(key)
        return entry

    def put(self, key: str, entry: np.ndarray) -> None:
        self._data[key] = entry
        self._data.move_to_end(key)
        while len(self._data) > self._maxsize:
            self._data.popitem(last=False)


class FilamentCrops(Dataset):
    \"\"\"Random crops from cached flattened images, biased toward filaments.

    ``target="annotator"`` (the default) trains against whichever single
    annotator's view was selected -- the original behaviour. ``target=
    "consensus"`` trains against ``scripts/preprocess.py``'s per-stem soft
    consensus mask instead: MAGFiLO's measured inter-annotator Panoptic
    Quality is only 0.343, so one annotator's view carries a lot of
    annotator-specific noise that a consensus average regresses out. The
    consensus mask is keyed by observation stem, not by view, so under this
    target ``image_ids`` is deduplicated to one view per stem -- the target
    no longer depends on which view was picked, only the crop geometry does.
    ``target="union"`` is the same mask cut at zero: a pixel is filament if
    any annotator drew it.

    ``preload=True`` decodes every flat image once, up front (about 4 MB
    each), instead of once per crop. Decoding is most of the time a crop
    takes, so this is what lets two training runs share a 4-CPU machine.
    DataLoader workers forked after that share the arrays rather than copy
    them; under the "spawn" start method (macOS) each worker would get its
    own copy, so leave it off there.
    \"\"\"

    def __init__(
        self,
        image_ids: Sequence[str],
        annotations: Annotations,
        cache_dir: str | Path,
        crop_size: int = 512,
        crops_per_image: int = 8,
        filament_bias: float = 0.7,
        augment: bool = True,
        seed: int = 0,
        target: str = "annotator",
        preload: bool = False,
    ) -> None:
        if target not in ("annotator", "consensus", "union"):
            raise ValueError(f"unknown target: {target!r}")

        self.annotations = annotations
        self.cache_dir = Path(cache_dir)
        self.crop_size = crop_size
        self.crops_per_image = crops_per_image
        self.filament_bias = filament_bias
        self.augment = augment
        self.seed = seed
        self.target = target

        image_ids = list(image_ids)
        if target in ("consensus", "union"):
            image_ids = deduplicate_by_file(annotations, image_ids, seed=seed)
        self.image_ids = image_ids

        self._flat_dir = self.cache_dir / "flat"
        self._mask_dir = self.cache_dir / "mask"
        self._consensus_dir = self.cache_dir / "consensus"
        self._disk = load_disk_geometry(self.cache_dir / "disk.json")
        self._centroids: dict[str, list[tuple[float, float]]] = {
            image_id: _instance_centroids(annotations, image_id) for image_id in self.image_ids
        }

        self._cache = _BoundedCache()
        self._preloaded: dict[str, np.ndarray] = {}
        if preload:
            for stem in sorted({annotations.images[i].stem for i in self.image_ids}):
                self._preloaded[stem] = self._read_flat(stem)
        # One RNG per DataLoader worker process -- see _rng().
        self._rngs: dict[int, random.Random] = {}

    def __len__(self) -> int:
        return len(self.image_ids) * self.crops_per_image

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image_id = self.image_ids[index // self.crops_per_image]
        record = self.annotations.images[image_id]
        stem = record.stem
        shape = (record.height, record.width)
        rng = self._rng()

        flat_u8 = self._flat(stem)
        y0, x0 = self._sample_crop_origin(rng, image_id, stem, shape)
        size = self.crop_size

        flat_crop = flat_u8[y0 : y0 + size, x0 : x0 + size].astype(np.float32) / 255.0
        radius_crop = self._disk[stem].radius_window(y0, x0, *flat_crop.shape).astype(np.float32)
        mask_crop = self._load_mask(image_id, stem, shape)[
            y0 : y0 + size, x0 : x0 + size
        ].astype(np.float32)

        if self.augment:
            out = _AUGMENT(image=flat_crop, mask=mask_crop, radius=radius_crop)
            flat_crop, mask_crop, radius_crop = out["image"], out["mask"], out["radius"]

        x = np.stack([flat_crop, radius_crop], axis=0).astype(np.float32)
        y = mask_crop[None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)

    # --- internals -----------------------------------------------------

    def _rng(self) -> random.Random:
        \"\"\"Per-worker RNG.

        ``self`` -- and any ``random.Random`` already stored on it -- is
        copied wholesale into each DataLoader worker process at fork time;
        keying by worker id and creating the generator lazily means sibling
        workers don't replay an identical crop sequence.
        \"\"\"
        worker = torch.utils.data.get_worker_info()
        key = 0 if worker is None else worker.id + 1
        rng = self._rngs.get(key)
        if rng is None:
            rng = random.Random(self.seed + key)
            self._rngs[key] = rng
        return rng

    def _flat(self, stem: str) -> np.ndarray:
        flat_u8 = self._preloaded.get(stem)
        if flat_u8 is None:
            flat_u8 = self._cache.get(stem)
        if flat_u8 is None:
            flat_u8 = self._read_flat(stem)
            self._cache.put(stem, flat_u8)
        return flat_u8

    def _read_flat(self, stem: str) -> np.ndarray:
        flat_path = self._flat_dir / f"{stem}.png"
        flat_u8 = cv2.imread(str(flat_path), cv2.IMREAD_GRAYSCALE)
        if flat_u8 is None:
            raise FileNotFoundError(
                f"missing cached flat image: {flat_path} -- run scripts/preprocess.py first"
            )
        return flat_u8

    def _load_mask(self, image_id: str, stem: str, shape: tuple[int, int]) -> np.ndarray:
        if self.target in ("consensus", "union"):
            consensus_path = self._consensus_dir / f"{stem}.png"
            mask = cv2.imread(str(consensus_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise FileNotFoundError(
                    f"missing cached consensus mask: {consensus_path} -- "
                    "run scripts/preprocess.py first"
                )
            if self.target == "union":
                return (mask > 0).astype(np.uint8)
            # Soft agreement fraction, not thresholded -- 255 means every
            # annotator of this stem agreed, not just a majority.
            return mask.astype(np.float32) / 255.0

        mask_path = self._mask_dir / f"{image_id}.png"
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            # Every one of the 1154 training views has at least one annotated
            # filament, so a missing mask is always a cache fault (a --limit
            # preprocessing run, a wiped directory). Treating it as "no
            # filaments" would silently teach the model to predict nothing.
            raise FileNotFoundError(
                f"missing cached mask: {mask_path} -- run scripts/preprocess.py first"
            )
        return (mask > 0).astype(np.uint8)

    def _sample_crop_origin(
        self, rng: random.Random, image_id: str, stem: str, shape: tuple[int, int]
    ) -> tuple[int, int]:
        \"\"\"Pick the crop's top-left corner, in image coordinates.\"\"\"
        height, width = shape
        size = self.crop_size
        centroids = self._centroids[image_id]

        if centroids and rng.random() < self.filament_bias:
            cy, cx = rng.choice(centroids)
            jitter = size / 4.0
            cy += rng.uniform(-jitter, jitter)
            cx += rng.uniform(-jitter, jitter)
        else:
            # Uniform over the disk's *area*, not its bounding box -- sampling
            # (angle, radius) uniformly would crowd crops toward the centre.
            disk = self._disk[stem]
            angle = rng.uniform(0.0, 2.0 * np.pi)
            r = disk.radius * (rng.random() ** 0.5)
            cy = disk.cy + r * np.sin(angle)
            cx = disk.cx + r * np.cos(angle)

        y0 = int(np.clip(cy - size / 2.0, 0, max(height - size, 0)))
        x0 = int(np.clip(cx - size / 2.0, 0, max(width - size, 0)))
        return y0, x0

""",
    'filament_seg/disk.py': """\"\"\"Solar disk geometry and limb-darkening correction.

Two preprocessing steps that matter more than the choice of model:

1. **Disk masking.** Everything outside the solar limb is sky and detector
   noise. It is uniformly dark, so any "filaments are dark" rule fires on it.
   Masking it away removes a large class of false positives for free -- and
   under Panoptic Quality every false positive costs 0.5 in the denominator.

2. **Limb-darkening correction.** A full-disk H-alpha image is systematically
   darker toward the limb, so a single global darkness threshold is far too
   eager near the edge and far too shy at disk centre. Dividing by the median
   radial intensity profile makes one threshold mean the same thing everywhere.
\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np


@dataclass(frozen=True)
class Disk:
    cx: float
    cy: float
    radius: float

    def mask(self, shape: tuple[int, int], shrink: float = 0.98) -> np.ndarray:
        \"\"\"Boolean mask of the disk, shrunk slightly to drop the noisy limb.\"\"\"
        height, width = shape
        ys, xs = np.ogrid[:height, :width]
        r2 = (xs - self.cx) ** 2 + (ys - self.cy) ** 2
        return r2 <= (self.radius * shrink) ** 2

    def radius_map(self, shape: tuple[int, int]) -> np.ndarray:
        \"\"\"Normalised radial distance (1.0 at the limb).\"\"\"
        return self.radius_window(0, 0, *shape)

    def radius_window(self, y0: int, x0: int, height: int, width: int) -> np.ndarray:
        \"\"\"``radius_map`` for rows ``y0:y0+height`` and columns ``x0:x0+width`` only.\"\"\"
        ys, xs = np.ogrid[y0 : y0 + height, x0 : x0 + width]
        return np.sqrt((xs - self.cx) ** 2 + (ys - self.cy) ** 2) / max(self.radius, 1.0)


def read_image(path: str | Path) -> np.ndarray:
    \"\"\"Load a GONG H-alpha JPEG as a single-channel uint8 array.

    The competition data is grayscale; reading it as BGR and averaging would
    only add rounding noise.
    \"\"\"
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"could not read image: {path}")
    return image


#: GONG H-alpha frames are usually registered to a standard geometry: the
#: disk centred and about 0.439 frame-widths in radius (899 px at 2048). Many
#: frames are not, so this is only one of several starting guesses.
_STANDARD_RADIUS_FRACTION = 0.439
#: Rays cast from a candidate centre when locating the limb.
_N_RAYS = 720
#: How far inside each starting guess to look as well, in px at 2048. The halo
#: can reach 150 px beyond the limb, so a guess locked onto it needs this.
_INWARD_OFFSETS = (0.0, 50.0, 100.0, 150.0)
#: A ray "agrees" with a circle if its sharpest edge lies within this many px
#: (at 2048) of it. Fixed, not data-adaptive, so scattered edges score low.
_AGREEMENT_TOLERANCE = 3.0
#: A circle counts as well supported if this fraction of the best circle's
#: agreeing rays also agree with it.
_SUPPORT_RATIO = 0.8


def detect_disk(image: np.ndarray) -> Disk:
    \"\"\"Locate the solar disk by its limb.

    GONG frames carry a smooth "halo" annulus outside the limb -- scattered
    light, and the area the frame covered before it was registered -- that can
    be nearly as bright as the disk itself, and its outer border is a crisp
    circle too. Thresholding the image, which this function used to rely on
    alone, locks onto that border: about 35 px beyond the limb on a typical
    frame and up to 150 px on the worst.

    Instead, several starting circles (an intensity threshold, a texture
    threshold -- the disk is textured, the halo smooth -- and the standard GONG
    geometry), each also shrunk by up to 150 px, are snapped to the sharpest
    radial edge nearby and scored by how many rays agree with the result. The
    halo always lies outside the limb, so of the well-supported circles the
    smallest is the limb.
    \"\"\"
    scale = min(image.shape) / 2048.0
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), max(1.5 * scale, 0.8))
    seeds = [_threshold_disk(image), _texture_disk(image), _standard_disk(image)]

    fits: list[tuple[Disk, float]] = []
    for seed in seeds:
        if seed is None:
            continue
        for offset in _INWARD_OFFSETS:
            start = Disk(seed.cx, seed.cy, seed.radius - offset * scale)
            if start.radius <= 0:
                continue
            disk, _ = _refine_limb(blurred, start, inside=60.0, outside=20.0)
            # A second, narrow pass lets the centre settle once the radius is close.
            fits.append(_refine_limb(blurred, disk, inside=10.0, outside=10.0))

    best_support = max((support for _, support in fits), default=0.0)
    # Fewer than half the rays agreeing means nothing found a clean limb; the
    # plain threshold circle is then the least surprising answer.
    if best_support < 0.5:
        return seeds[0]
    supported = [disk for disk, support in fits if support >= _SUPPORT_RATIO * best_support]
    return min(supported, key=lambda disk: disk.radius)


def _fit_circle(xs: np.ndarray, ys: np.ndarray, iterations: int = 4) -> tuple[Disk, np.ndarray]:
    \"\"\"Least-squares (Kasa) circle fit, refitted without outliers.

    Returns the circle and a boolean inlier mask over the input points.
    \"\"\"
    keep = np.ones(xs.size, dtype=bool)
    disk = Disk(0.0, 0.0, 1.0)
    for _ in range(iterations):
        if keep.sum() < 3:
            break
        design = np.column_stack([xs[keep], ys[keep], np.ones(int(keep.sum()))])
        target = -(xs[keep] ** 2 + ys[keep] ** 2)
        d, e, f = np.linalg.lstsq(design, target, rcond=None)[0]
        cx, cy = -d / 2.0, -e / 2.0
        disk = Disk(float(cx), float(cy), float(np.sqrt(max(cx * cx + cy * cy - f, 1.0))))
        residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius
        spread = 1.4826 * np.median(np.abs(residual[keep] - np.median(residual[keep])))
        keep = np.abs(residual) <= max(3.0 * spread, 2.0)
    return disk, keep


def _refine_limb(
    blurred: np.ndarray, start: Disk, inside: float, outside: float
) -> tuple[Disk, float]:
    \"\"\"Snap ``start`` to the sharpest radial edge within the search band.

    ``blurred`` is the lightly smoothed float image. Band lengths are in px at
    2048 and scale with the frame, so the same code works on small test
    images. Returns the refined circle and the fraction of all rays whose
    sharpest edge lies within ``_AGREEMENT_TOLERANCE`` of it.
    \"\"\"
    height, width = blurred.shape
    scale = min(height, width) / 2048.0
    theta = np.linspace(0.0, 2.0 * np.pi, _N_RAYS, endpoint=False)[:, None]
    radii = np.arange(start.radius - inside * scale, start.radius + outside * scale + 1.0, 1.0)
    map_x = (start.cx + radii[None, :] * np.cos(theta)).astype(np.float32)
    map_y = (start.cy + radii[None, :] * np.sin(theta)).astype(np.float32)
    profiles = cv2.remap(blurred, map_x, map_y, cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    in_frame = (map_x >= 1) & (map_x <= width - 2) & (map_y >= 1) & (map_y <= height - 2)
    gradient = np.abs(np.gradient(profiles, axis=1))
    gradient[~in_frame] = -1.0
    edge = np.argmax(gradient, axis=1)
    valid = gradient[np.arange(_N_RAYS), edge] > 0
    # A real edge is a peak inside the band. A maximum on the band's boundary
    # is just the smooth limb-darkening slope rising toward it -- perfectly
    # circular, so without this it would pass for a well-supported limb.
    valid &= (edge > 1) & (edge < radii.size - 2)
    if valid.sum() < 3:
        return start, 0.0
    # Sub-pixel peak position from a parabola through the peak and its two
    # neighbours: the flattening downstream is sensitive to fractions of a
    # pixel in the radius.
    rows = np.flatnonzero(valid)
    peak = edge[valid]
    before, at, after = (gradient[rows, peak - 1], gradient[rows, peak],
                         gradient[rows, peak + 1])
    curvature = before - 2.0 * at + after
    shift = np.divide(0.5 * (before - after), curvature,
                      out=np.zeros_like(at), where=curvature < 0)
    r = radii[peak] + np.clip(shift, -0.5, 0.5)
    xs = start.cx + r * np.cos(theta[valid, 0])
    ys = start.cy + r * np.sin(theta[valid, 0])
    disk, _ = _fit_circle(xs, ys)
    residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius
    agreeing = np.abs(residual) <= max(_AGREEMENT_TOLERANCE * scale, 1.0)
    return disk, float(agreeing.sum()) / _N_RAYS


def _standard_disk(image: np.ndarray) -> Disk:
    height, width = image.shape
    return Disk((width - 1) / 2.0, height / 2.0, min(height, width) * _STANDARD_RADIUS_FRACTION)


def _texture_disk(image: np.ndarray) -> Disk | None:
    \"\"\"Disk from local texture: chromospheric structure is busy, the halo smooth.\"\"\"
    scale = min(image.shape) / 2048.0
    img = image.astype(np.float32)
    detail = np.abs(img - cv2.GaussianBlur(img, (0, 0), max(3.0 * scale, 1.0)))
    texture = cv2.GaussianBlur(detail, (0, 0), max(15.0 * scale, 2.0))
    texture_u8 = np.clip(texture * (255.0 / max(float(texture.max()), 1e-6)), 0, 255)
    _, binary = cv2.threshold(texture_u8.astype(np.uint8), 0, 1,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n_labels <= 1:
        return None
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    component = (labels == largest).astype(np.uint8)
    # Fill the component's interior: smooth patches on the disk (and the whole
    # interior of a texture-free test image) leave holes.
    flood = component.copy()
    cv2.floodFill(flood, np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), np.uint8), (0, 0), 1)
    component = component | (1 - flood)
    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    points = max(contours, key=cv2.contourArea)[:, 0, :].astype(np.float64)
    if len(points) < 3:
        return None
    disk, _ = _fit_circle(points[:, 0], points[:, 1])
    return disk


def _threshold_disk(image: np.ndarray) -> Disk:
    \"\"\"Disk from an intensity threshold: the brightest large component.

    Reliable for the centre of a clean frame, but the halo annulus usually
    passes the threshold too, so the radius it gives is an upper bound.
    \"\"\"
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=5)
    threshold, binary = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    binary = cv2.morphologyEx(
        binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    )

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n_labels <= 1:
        # Degenerate image: fall back to the nominal full-frame disk.
        height, width = image.shape
        return Disk(width / 2, height / 2, min(height, width) * 0.45)

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    component = (labels == largest).astype(np.uint8)

    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(contour)
    return Disk(float(cx), float(cy), float(radius))


def limb_darkening_profile(
    image: np.ndarray, disk: Disk, n_bins: int = 128, smooth: int = 9
) -> np.ndarray:
    \"\"\"Median intensity as a function of normalised radius.\"\"\"
    radius = disk.radius_map(image.shape)
    inside = radius <= 1.0
    bins = np.clip((radius[inside] * n_bins).astype(np.int32), 0, n_bins - 1)
    values = image[inside].astype(np.float32)

    profile = np.ones(n_bins, dtype=np.float32)
    order = np.argsort(bins, kind="stable")
    bins_sorted, values_sorted = bins[order], values[order]
    edges = np.searchsorted(bins_sorted, np.arange(n_bins + 1))
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        if hi > lo:
            profile[b] = np.median(values_sorted[lo:hi])

    # Fill empty bins, then smooth so the correction has no step artefacts.
    valid = profile > 0
    if valid.any():
        profile = np.interp(np.arange(n_bins), np.flatnonzero(valid), profile[valid])
    if smooth > 1:
        kernel = np.ones(smooth, dtype=np.float32) / smooth
        profile = np.convolve(profile, kernel, mode="same")
    profile[profile <= 0] = 1.0
    return profile


def flatten(image: np.ndarray, disk: Disk, n_bins: int = 128) -> np.ndarray:
    \"\"\"Divide out limb darkening.

    Returns a float32 image where 1.0 is "typical quiet Sun at this radius" and
    filaments sit clearly below 1.0, independent of distance from disk centre.
    \"\"\"
    profile = limb_darkening_profile(image, disk, n_bins=n_bins)
    radius = disk.radius_map(image.shape)
    bin_index = np.clip((radius * n_bins).astype(np.int32), 0, n_bins - 1)
    expected = profile[bin_index]
    flat = image.astype(np.float32) / np.maximum(expected, 1e-3)
    flat[radius > 1.0] = 1.0
    return flat

""",
    'filament_seg/fusion.py': """\"\"\"Detector-guided instance formation.

The U-Net answers "is this pixel a filament?" well -- its matched masks reach
an IoU of about 0.67 -- but it has no notion of which pixels belong to the
*same* filament. Connected components and gap bridging guess that from
geometry alone, and that guess is where most of the Panoptic Quality goes:
the U-Net pipeline makes about as many false positives as true positives. An
instance detector answers the grouping question directly and scores every
instance, but its masks are coarse: predicted at a quarter of its input
resolution, which is itself half the image's.

Fusion takes the best of both. Each detection, in descending confidence,
claims the full-resolution U-Net filament pixels inside a slightly grown copy
of its coarse mask. A pixel belongs to the first detection that claims it, so
instances never overlap. U-Net pixels that no detection claims are dropped --
or, with ``keep_unclaimed``, grouped the way the U-Net-only pipeline groups
them (gap bridging, then an area cut), for filaments the detector missed. At
``keep_unclaimed`` equal to the U-Net pipeline's own ``min_area``, fusion only
changes the grouping where a detection is confident enough to claim pixels.
\"\"\"

from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np

from .postprocess import (
    PostprocessParams,
    _ellipse,
    bridge_and_label,
    clean_binary,
    fill_holes,
    probability_to_logit,
)
from pycocotools import mask as mask_utils

from .rle import counts_to_mask, rle_from_counts


@dataclass(frozen=True)
class Detection:
    \"\"\"One detector instance: its confidence and its mask as COCO RLE counts.\"\"\"

    score: float
    counts: str


@dataclass(frozen=True)
class Region:
    \"\"\"A detection's grown mask, cropped to its bounding box.\"\"\"

    score: float
    y0: int
    y1: int
    x0: int
    x1: int
    mask: np.ndarray


@dataclass
class FusionParams:
    \"\"\"Knobs for detector-guided instance formation.\"\"\"

    #: U-Net probability above which a pixel counts as filament.
    threshold: float = 0.5
    #: Morphological closing applied to the U-Net mask before assignment.
    close_radius: int = 3
    #: Pixels the detector's coarse mask is grown by before claiming U-Net
    #: pixels; it is predicted at an eighth of full resolution.
    grow: int = 8
    #: Detections below this confidence claim nothing.
    min_score: float = 0.25
    #: Instances smaller than this, after claiming, are dropped.
    min_area: int = 400
    #: Unclaimed U-Net pixels are grouped by bridging gaps of this many
    #: pixels, and groups at least ``keep_unclaimed`` large are kept as
    #: instances of their own; 0 drops every unclaimed pixel.
    keep_unclaimed: int = 0
    unclaimed_gap: int = 24


def unet_binary(
    logits: np.ndarray, disk_mask: np.ndarray, threshold: float, close_radius: int
) -> np.ndarray:
    \"\"\"The U-Net's filament pixels: thresholded, closed, holes filled, on the disk.\"\"\"
    binary = ((logits > probability_to_logit(threshold)) & disk_mask).astype(np.uint8)
    binary = clean_binary(binary, PostprocessParams(open_radius=0, close_radius=close_radius))
    return fill_holes(binary).astype(bool) & disk_mask


def grow_regions(
    detections: list[Detection],
    grow: int,
    shape: tuple[int, int],
    min_score: float = 0.0,
) -> list[Region]:
    \"\"\"Decode, crop and grow each detection's mask, highest confidence first.\"\"\"
    height, width = shape
    regions = []
    kernel = _ellipse(grow) if grow > 0 else None
    for detection in sorted(detections, key=lambda d: -d.score):
        if detection.score < min_score:
            continue
        mask = counts_to_mask(detection.counts, height, width)
        ys, xs = np.nonzero(mask)
        if ys.size == 0:
            continue
        y0, y1 = max(int(ys.min()) - grow, 0), min(int(ys.max()) + grow + 1, height)
        x0, x1 = max(int(xs.min()) - grow, 0), min(int(xs.max()) + grow + 1, width)
        crop = mask[y0:y1, x0:x1].astype(np.uint8)
        if kernel is not None:
            crop = cv2.dilate(crop, kernel)
        regions.append(Region(detection.score, y0, y1, x0, x1, crop.astype(bool)))
    return regions


def assign(
    binary: np.ndarray,
    regions: list[Region],
    min_score: float,
    min_area: int,
    keep_unclaimed: int = 0,
    unclaimed_gap: int = 24,
) -> np.ndarray:
    \"\"\"Integer instance labels: each region, best first, claims its U-Net pixels.\"\"\"
    labels = np.zeros(binary.shape, dtype=np.int32)
    claimed = np.zeros(binary.shape, dtype=bool)
    next_label = 1
    for region in regions:
        if region.score < min_score:
            break
        window = (slice(region.y0, region.y1), slice(region.x0, region.x1))
        pixels = binary[window] & region.mask & ~claimed[window]
        if int(pixels.sum()) < min_area:
            continue
        labels[window][pixels] = next_label
        claimed[window] |= pixels
        next_label += 1

    if keep_unclaimed > 0:
        components = bridge_and_label((binary & ~claimed).astype(np.uint8), unclaimed_gap)
        counts = np.bincount(components.ravel())
        keep = counts >= keep_unclaimed
        keep[0] = False
        lookup = np.zeros(counts.size, dtype=np.int32)
        lookup[keep] = np.arange(next_label, next_label + int(keep.sum()), dtype=np.int32)
        extra = lookup[components]
        labels = np.where(extra > 0, extra, labels)
    return labels


def fuse(
    logits: np.ndarray,
    disk_mask: np.ndarray,
    detections: list[Detection],
    params: FusionParams | None = None,
) -> np.ndarray:
    \"\"\"U-Net logits + detector instances -> integer instance label map.\"\"\"
    params = params or FusionParams()
    binary = unet_binary(logits, disk_mask, params.threshold, params.close_radius)
    regions = grow_regions(detections, params.grow, logits.shape, params.min_score)
    return assign(binary, regions, params.min_score, params.min_area, params.keep_unclaimed,
                  params.unclaimed_gap)


def merge_views(views: list[list[Detection]], iou_threshold: float = 0.5) -> list[Detection]:
    \"\"\"Merge detections from flipped copies of one image into one list.

    ``views`` holds each copy's detections, already flipped back. Detections
    from different copies that overlap with IoU above ``iou_threshold`` are
    the same filament; it keeps the best member's mask and scores the
    average over *all* copies, counting a copy that missed it as 0. A
    filament the detector finds from every orientation keeps its confidence;
    one it sees in a single orientation loses most of it.
    \"\"\"
    pool = sorted(((det, view) for view, dets in enumerate(views) for det in dets),
                  key=lambda item: -item[0].score)
    if not pool:
        return []
    rles = [rle_from_counts(det.counts) for det, _ in pool]
    ious = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)))
    used = np.zeros(len(pool), dtype=bool)
    merged = []
    for i, (detection, view) in enumerate(pool):
        if used[i]:
            continue
        used[i] = True
        seen, total = {view}, detection.score
        for j in np.argsort(-ious[i], kind="stable"):
            if ious[i, j] <= iou_threshold:
                break
            if used[j] or pool[j][1] in seen:
                continue
            used[j] = True
            seen.add(pool[j][1])
            total += pool[j][0].score
        merged.append(Detection(total / len(views), detection.counts))
    return merged

""",
    'filament_seg/logit_cache.py': """\"\"\"A disk cache of full-resolution model logits, safe against staleness.

Tiled inference with test-time augmentation is the expensive half of every
evaluation, and it does not depend on any post-processing setting, so its
output -- one 2048x2048 logit map per observation, float16 (8 MB) -- is cached
and reused by every grid point and every later experiment. The cache is keyed
by observation name alone, so it also records which model, TTA mode and tiling
produced it and refuses to mix them.
\"\"\"

from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import torch

from .dataset import load_model_input
from .disk import Disk
from .model import ensemble_predict, load_trained


def cache_path(logit_dir: Path, stem: str) -> Path:
    return logit_dir / f"{stem}.npy"


def check_cache_provenance(
    logit_dir: Path,
    checkpoint_paths: list[str],
    tta: str,
    tile: int,
    overlap: int,
    force: bool,
) -> None:
    \"\"\"Make sure every cached logit map was produced by this model and setup.

    The cache is keyed by observation name alone, so without this a sweep run
    against a new checkpoint, TTA mode or tiling would silently reuse stale
    logits and tune against the wrong predictions. ``force`` deletes the old
    maps *before* recording the new provenance: otherwise a forced run that
    died part-way would leave the previous model's maps under the new record,
    and the next plain run would accept them.
    \"\"\"
    checkpoints = []
    for path in checkpoint_paths:
        resolved = Path(path).resolve()
        stat = resolved.stat()
        checkpoints.append({"path": str(resolved), "size": stat.st_size,
                            "mtime": int(stat.st_mtime)})
    wanted = {"checkpoints": checkpoints, "tta": tta, "tile": tile, "overlap": overlap}

    logit_dir.mkdir(parents=True, exist_ok=True)
    meta_path = logit_dir / "cache_meta.json"
    cached = sorted(logit_dir.glob("*.npy"))
    if force:
        for path in cached:
            path.unlink()
    elif meta_path.exists():
        found = json.loads(meta_path.read_text(encoding="utf-8"))
        if found != wanted:
            raise SystemExit(
                f"{logit_dir} holds logits from a different model or setup:\\n"
                f"  cached: {found}\\n  wanted: {wanted}\\n"
                "use a different --out-dir, or --force to recompute"
            )
    elif cached:
        raise SystemExit(
            f"{logit_dir} holds cached logits with no record of which model made them; "
            "use a different --out-dir, or --force to recompute"
        )
    meta_path.write_text(json.dumps(wanted, indent=2), encoding="utf-8")


def ensure_logits(
    stems: list[str],
    checkpoint_paths: list[str],
    disk_geometry: dict[str, Disk],
    flat_dir: Path,
    logit_dir: Path,
    tile: int,
    overlap: int,
    device: torch.device,
    force: bool,
    tta: str = "none",
) -> None:
    \"\"\"Cache the (ensemble-averaged, TTA-averaged) logits for every stem, once.\"\"\"
    check_cache_provenance(logit_dir, checkpoint_paths, tta, tile, overlap, force)
    pending = [s for s in stems if not cache_path(logit_dir, s).exists()]
    if not pending:
        print(f"logits: {len(stems)}/{len(stems)} already cached, skipping model load")
        return

    print(f"device: {device}")
    models = [load_trained(path, device) for path in checkpoint_paths]
    print(f"models: {len(models)}; computing logits for {len(pending)}/{len(stems)} observations")
    for n, stem in enumerate(pending, start=1):
        x = load_model_input(flat_dir, stem, disk_geometry[stem])
        if x is None:
            raise SystemExit(f"no cached flat image for {stem} -- run scripts/preprocess.py")
        logits = ensemble_predict(models, x, tta=tta, tile=tile, overlap=overlap, device=device)
        # Written under a temporary name and renamed, so an interrupted run can
        # never leave a truncated map that a later run would take as complete.
        partial = logit_dir / f"{stem}.npy.partial"
        with open(partial, "wb") as handle:
            np.save(handle, logits.astype(np.float16))
        os.replace(partial, cache_path(logit_dir, stem))
        if n % 5 == 0 or n == len(pending):
            print(f"  {n}/{len(pending)}", flush=True)

""",
    'filament_seg/metrics.py': """\"\"\"Local re-implementation of the competition's evaluation.

Leaderboard metric (competition Overview -> "Leaderboard Ranking"):

    PQ(Y, Y_hat) = sum_{(y, y_hat) in TP} IoU(y, y_hat)
                   / (|TP| + 0.5 |FP| + 0.5 |FN|)

which is exactly standard Panoptic Quality (Kirillov et al., CVPR 2019) with the
usual IoU > 0.5 matching rule. Above 0.5 the match is provably unique, so greedy
assignment by descending IoU is optimal.

Two design notes:

* **Aggregation is pooled.** The organisers' self-evaluation notebook
  (kaggle.com/code/azimahmadzadeh/self-evaluation-notebook) matches the
  predictions against every annotator's view of an observation separately and
  pools TP, FP, FN and the matched IoUs over all views of all observations;
  ``filament_seg.scoring`` keeps those totals per observation. ``summarize``
  reports the pooled score (``pq_pooled``) and, for diagnostics only, the mean
  per-image PQ (``pq_per_image_mean``).

* **The rubric wants more than PQ.** 70% of the final score is quantitative and
  explicitly includes the *distributions* of Dice and IoU plus the counts of
  one-to-many (fragmentation) and many-to-one (over-merge) relations. Those are
  produced here too, so the report can be written from real numbers.

Marginal-value rule worth remembering while tuning post-processing: a predicted
instance that fails to reach IoU 0.5 costs 0.5 FP + 0.5 FN = 1.0 in the
denominator, while simply not predicting it costs only 0.5. Emitting a doubtful
instance is *twice* as expensive as staying silent, so a prediction is only
worth making when its chance of clearing the threshold is above roughly
``PQ / (2 * E[IoU | hit])`` -- around 30% at a PQ of 0.4.
\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Sequence

import numpy as np

from .config import FRAGMENT_OVERLAP_THRESHOLD, PQ_IOU_THRESHOLD
from .rle import Rle, iou_matrix, rle_areas, rle_intersection_area


@dataclass
class ImageResult:
    \"\"\"Per-image matching outcome.\"\"\"

    image_id: str
    n_gt: int
    n_pred: int
    tp: int
    fp: int
    fn: int
    matched_ious: list[float] = field(default_factory=list)
    #: Ground-truth segments split across >= 2 predictions (fragmentation).
    one_to_many: int = 0
    #: Predictions covering >= 2 ground-truth segments (over-merge).
    many_to_one: int = 0

    @property
    def iou_sum(self) -> float:
        return float(sum(self.matched_ious))

    @property
    def denominator(self) -> float:
        return self.tp + 0.5 * self.fp + 0.5 * self.fn

    @property
    def pq(self) -> float:
        if self.denominator == 0:
            # No ground truth and no prediction: nothing to get wrong.
            return 1.0
        return self.iou_sum / self.denominator

    @property
    def matched_dices(self) -> list[float]:
        # Dice and IoU are algebraically linked: Dice = 2*IoU / (1 + IoU).
        return [2.0 * i / (1.0 + i) for i in self.matched_ious]


def match_instances(
    gt_rles: Sequence[Rle],
    pred_rles: Sequence[Rle],
    iou_threshold: float = PQ_IOU_THRESHOLD,
) -> tuple[list[tuple[int, int, float]], np.ndarray]:
    \"\"\"Greedily match predictions to ground truth by descending IoU.

    Returns ``(matches, iou_matrix)`` where each match is ``(gt_i, pred_j, iou)``.
    \"\"\"
    ious = iou_matrix(gt_rles, pred_rles)
    matches: list[tuple[int, int, float]] = []
    if ious.size == 0:
        return matches, ious

    candidates = np.argwhere(ious > iou_threshold)
    order = np.argsort([-ious[g, p] for g, p in candidates]) if len(candidates) else []
    used_gt: set[int] = set()
    used_pred: set[int] = set()
    for idx in order:
        g, p = candidates[idx]
        g, p = int(g), int(p)
        if g in used_gt or p in used_pred:
            continue
        used_gt.add(g)
        used_pred.add(p)
        matches.append((g, p, float(ious[g, p])))
    return matches, ious


def count_fragmentation(
    gt_rles: Sequence[Rle],
    pred_rles: Sequence[Rle],
    overlap_threshold: float = FRAGMENT_OVERLAP_THRESHOLD,
) -> tuple[int, int]:
    \"\"\"Count one-to-many and many-to-one relations.

    A ground-truth segment is *fragmented* when at least two predictions each
    cover more than ``overlap_threshold`` of its area. A prediction is an
    *over-merge* when it covers more than ``overlap_threshold`` of at least two
    ground-truth segments.
    \"\"\"
    if not gt_rles or not pred_rles:
        return 0, 0

    gt_area = rle_areas(gt_rles)
    inter = np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)
    ious = iou_matrix(gt_rles, pred_rles)
    for g in range(len(gt_rles)):
        for p in range(len(pred_rles)):
            if ious[g, p] > 0:
                inter[g, p] = rle_intersection_area(gt_rles[g], pred_rles[p])

    with np.errstate(divide="ignore", invalid="ignore"):
        coverage = np.where(gt_area[:, None] > 0, inter / gt_area[:, None], 0.0)

    significant = coverage > overlap_threshold
    one_to_many = int((significant.sum(axis=1) >= 2).sum())
    many_to_one = int((significant.sum(axis=0) >= 2).sum())
    return one_to_many, many_to_one


def evaluate_image(
    image_id: str,
    gt_rles: Sequence[Rle],
    pred_rles: Sequence[Rle],
    iou_threshold: float = PQ_IOU_THRESHOLD,
    with_fragmentation: bool = True,
) -> ImageResult:
    matches, _ = match_instances(gt_rles, pred_rles, iou_threshold)
    tp = len(matches)
    result = ImageResult(
        image_id=image_id,
        n_gt=len(gt_rles),
        n_pred=len(pred_rles),
        tp=tp,
        fp=len(pred_rles) - tp,
        fn=len(gt_rles) - tp,
        matched_ious=[m[2] for m in matches],
    )
    if with_fragmentation:
        result.one_to_many, result.many_to_one = count_fragmentation(gt_rles, pred_rles)
    return result


def summarize(results: Sequence[ImageResult]) -> dict:
    \"\"\"Aggregate per-image results into the numbers the rubric asks for.\"\"\"
    if not results:
        return {"n_images": 0}

    tp = sum(r.tp for r in results)
    fp = sum(r.fp for r in results)
    fn = sum(r.fn for r in results)
    iou_sum = sum(r.iou_sum for r in results)
    denom = tp + 0.5 * fp + 0.5 * fn

    all_ious = np.array([i for r in results for i in r.matched_ious], dtype=np.float64)
    all_dices = np.array([d for r in results for d in r.matched_dices], dtype=np.float64)
    per_image = np.array([r.pq for r in results], dtype=np.float64)

    def distribution(values: np.ndarray) -> dict:
        if values.size == 0:
            return {"n": 0}
        percentiles = np.percentile(values, [5, 25, 50, 75, 95])
        return {
            "n": int(values.size),
            "mean": float(values.mean()),
            "std": float(values.std()),
            "min": float(values.min()),
            "p05": float(percentiles[0]),
            "p25": float(percentiles[1]),
            "median": float(percentiles[2]),
            "p75": float(percentiles[3]),
            "p95": float(percentiles[4]),
            "max": float(values.max()),
        }

    return {
        "n_images": len(results),
        # The two candidate aggregations -- see the module docstring.
        "pq_pooled": float(iou_sum / denom) if denom else float("nan"),
        "pq_per_image_mean": float(per_image.mean()),
        # Standard PQ = SQ * RQ decomposition, useful for diagnosis:
        # SQ says "how good are the masks I got right", RQ says "how often am I right".
        "sq": float(iou_sum / tp) if tp else 0.0,
        "rq": float(tp / denom) if denom else 0.0,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "n_gt": sum(r.n_gt for r in results),
        "n_pred": sum(r.n_pred for r in results),
        "one_to_many": sum(r.one_to_many for r in results),
        "many_to_one": sum(r.many_to_one for r in results),
        "iou_distribution": distribution(all_ious),
        "dice_distribution": distribution(all_dices),
        "pq_per_image_distribution": distribution(per_image),
    }


def evaluate(
    gt: dict[str, Sequence[Rle]],
    pred: dict[str, Sequence[Rle]],
    iou_threshold: float = PQ_IOU_THRESHOLD,
    with_fragmentation: bool = True,
) -> tuple[dict, list[ImageResult]]:
    \"\"\"Score a prediction dict against a ground-truth dict, keyed by image.\"\"\"
    results = [
        evaluate_image(
            image_id,
            gt[image_id],
            pred.get(image_id, []),
            iou_threshold=iou_threshold,
            with_fragmentation=with_fragmentation,
        )
        for image_id in sorted(gt)
    ]
    # Predictions for images absent from the ground truth are pure false positives.
    for image_id in sorted(set(pred) - set(gt)):
        n_pred = len(pred[image_id])
        results.append(
            ImageResult(image_id=image_id, n_gt=0, n_pred=n_pred, tp=0, fp=n_pred, fn=0)
        )
    return summarize(results), results


def format_report(summary: dict) -> str:
    \"\"\"Human-readable summary for the terminal and the technical report.\"\"\"
    if not summary.get("n_images"):
        return "no images evaluated"

    def dist_line(name: str, dist: dict) -> str:
        if not dist.get("n"):
            return f"  {name:<6} (none)"
        return (
            f"  {name:<6} n={dist['n']:<5} mean={dist['mean']:.3f}  "
            f"p05={dist['p05']:.3f} p25={dist['p25']:.3f} med={dist['median']:.3f} "
            f"p75={dist['p75']:.3f} p95={dist['p95']:.3f}"
        )

    return "\\n".join(
        [
            f"images={summary['n_images']}  gt={summary['n_gt']}  pred={summary['n_pred']}",
            f"PQ (pooled)      {summary['pq_pooled']:.4f}",
            f"PQ (per-image)   {summary['pq_per_image_mean']:.4f}",
            f"SQ={summary['sq']:.4f}  RQ={summary['rq']:.4f}",
            f"TP={summary['tp']}  FP={summary['fp']}  FN={summary['fn']}",
            f"fragmentation: one-to-many={summary['one_to_many']}  "
            f"many-to-one={summary['many_to_one']}",
            "distributions over matched pairs:",
            dist_line("IoU", summary["iou_distribution"]),
            dist_line("Dice", summary["dice_distribution"]),
            dist_line("PQ/img", summary["pq_per_image_distribution"]),
        ]
    )

""",
    'filament_seg/model.py': """\"\"\"Segmentation network, loss, device selection and tiled inference.

A U-Net over a ResNet encoder is a deliberately boring choice: at native
2048x2048 resolution there is no budget for architecture search, and a
well-understood encoder/decoder pair with ImageNet-pretrained weights gives
the fastest path to a working model. The two-channel input (flattened
intensity + disk radius) needs its own first conv, so the pretrained weights
only warm-start the encoder trunk, not that very first layer.

``tiled_predict`` lives here rather than duplicated in ``scripts/train.py``
(per-epoch validation) and ``scripts/predict.py`` (submission generation) so
the two can never quietly drift apart on how tiles are blended.
\"\"\"

from __future__ import annotations

import segmentation_models_pytorch as smp
import numpy as np
import torch
import torch.nn as nn


def build_model(
    encoder: str = "resnet34",
    in_channels: int = 2,
    weights: str | None = "imagenet",
) -> nn.Module:
    \"\"\"U-Net over ``encoder``, one logit output channel (no activation).

    The loss (``DiceBCELoss``) applies ``BCEWithLogitsLoss`` internally, so
    the network stays activation-free -- this is also what keeps it
    numerically stable under autocast.
    \"\"\"
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights=weights,
        in_channels=in_channels,
        classes=1,
        activation=None,
    )


class DiceBCELoss(nn.Module):
    \"\"\"Soft Dice + BCE-with-logits.

    Filament pixels are roughly 0.2% of the disk, so plain BCE is dominated
    by the easy background majority long before its gradient starts pushing
    on filament boundaries. Dice is defined on the foreground overlap and is
    immune to how much background surrounds it; BCE is kept alongside it
    because Dice alone gives a noisy, sometimes unstable signal on crops with
    very few (or zero) positive pixels.

    ``fn_weight`` turns the Dice term into a Tversky index: above 0.5, missed
    filament pixels cost more than extra ones (Salehi et al., 2017). At 0.5,
    the default, it is exactly soft Dice.
    \"\"\"

    def __init__(
        self,
        dice_weight: float = 1.0,
        bce_weight: float = 1.0,
        smooth: float = 1.0,
        fn_weight: float = 0.5,
    ) -> None:
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.fn_weight = fn_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce_loss = self.bce(logits, target)

        probs = torch.sigmoid(logits).flatten(1)
        target_flat = target.flatten(1)
        intersection = (probs * target_flat).sum(dim=1)
        false_pos = probs.sum(dim=1) - intersection
        false_neg = target_flat.sum(dim=1) - intersection
        denominator = (2.0 * intersection + 2.0 * (1.0 - self.fn_weight) * false_pos
                       + 2.0 * self.fn_weight * false_neg)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        dice_loss = 1.0 - dice.mean()

        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


def select_device() -> torch.device:
    \"\"\"Prefer CUDA, then Apple MPS, then CPU.\"\"\"
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _hann_window(size: int) -> np.ndarray:
    window = np.hanning(size)
    if window.sum() == 0:  # size == 1: np.hanning gives an all-zero window
        window = np.ones(size)
    return np.outer(window, window).astype(np.float32)


@torch.no_grad()
def tiled_predict(
    model: nn.Module,
    x: np.ndarray,
    tile: int = 512,
    overlap: int = 128,
    device: torch.device | None = None,
    batch_size: int = 4,
) -> np.ndarray:
    \"\"\"Run ``model`` over overlapping tiles of a full-resolution image.

    Tiles are blended with a 2D Hann window rather than averaged uniformly: a
    tile's prediction is least reliable right at its own edge (no context
    beyond the tile boundary), so down-weighting the edges is what keeps the
    seams between tiles from fragmenting a filament that crosses one -- and
    fragmentation is punished roughly three times over under Panoptic
    Quality (see ``filament_seg.postprocess``).

    ``x`` is ``(C, H, W)`` float32; returns full-resolution logits ``(H, W)``.
    \"\"\"
    model.eval()
    device = device or next(model.parameters()).device
    _, height, width = x.shape
    stride = max(tile - overlap, 1)

    def positions(extent: int) -> list[int]:
        starts = list(range(0, max(extent - tile, 0) + 1, stride))
        if not starts or starts[-1] != extent - tile:
            starts.append(max(extent - tile, 0))
        return starts

    ys, xs = positions(height), positions(width)
    window = _hann_window(tile)
    logits_sum = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    grid = [(y, xc) for y in ys for xc in xs]
    for start in range(0, len(grid), batch_size):
        batch = grid[start : start + batch_size]
        tiles = np.stack([x[:, y : y + tile, xc : xc + tile] for y, xc in batch], axis=0)
        tensor = torch.from_numpy(tiles).to(device)
        out = model(tensor).squeeze(1).float().cpu().numpy()
        for (y, xc), tile_logits in zip(batch, out):
            logits_sum[y : y + tile, xc : xc + tile] += tile_logits * window
            weight_sum[y : y + tile, xc : xc + tile] += window

    weight_sum[weight_sum == 0] = 1.0
    return logits_sum / weight_sum


#: Test-time augmentation: which orientations to average over, each written as
#: (flip left-right, flip up-down, transpose). Training augmented with flips and
#: 90-degree rotations, so the model has seen all eight orientations of a square
#: image ("dihedral"); "flips" is the cheaper half of them.
TTA_MODES: dict[str, list[tuple[bool, bool, bool]]] = {
    "none": [(False, False, False)],
    "flips": [(False, False, False), (True, False, False),
              (False, True, False), (True, True, False)],
    "dihedral": [(flip_lr, flip_ud, transpose)
                 for transpose in (False, True)
                 for flip_lr in (False, True)
                 for flip_ud in (False, True)],
}


def _orient(a: np.ndarray, flip_lr: bool, flip_ud: bool, transpose: bool) -> np.ndarray:
    \"\"\"Apply one orientation to the last two (spatial) axes.\"\"\"
    if transpose:
        a = np.swapaxes(a, -1, -2)
    if flip_lr:
        a = a[..., :, ::-1]
    if flip_ud:
        a = a[..., ::-1, :]
    return np.ascontiguousarray(a)


def _unorient(a: np.ndarray, flip_lr: bool, flip_ud: bool, transpose: bool) -> np.ndarray:
    \"\"\"Undo :func:`_orient` on a 2-D map -- the same steps, in reverse order.\"\"\"
    if flip_ud:
        a = a[::-1, :]
    if flip_lr:
        a = a[:, ::-1]
    if transpose:
        a = a.T
    return a


@torch.no_grad()
def tiled_predict_tta(
    model: nn.Module, x: np.ndarray, tta: str = "none", **tiled_kwargs
) -> np.ndarray:
    \"\"\"``tiled_predict`` averaged over mirrored and rotated copies of the image.

    Each copy is predicted, turned back to the original orientation, and the
    logits are averaged. The prediction for a thin filament then no longer
    depends on which way the tiles happened to cut across it. Both input
    channels -- intensity and the radius map -- are transformed together, so the
    model always sees a geometrically consistent image.

    Averaging makes logits less extreme than a single pass, so a threshold tuned
    without TTA is not automatically right with it.
    \"\"\"
    orientations = TTA_MODES[tta]
    total: np.ndarray | None = None
    for orientation in orientations:
        logits = tiled_predict(model, _orient(x, *orientation), **tiled_kwargs)
        logits = _unorient(logits, *orientation)
        total = logits.copy() if total is None else total + logits
    return total / len(orientations)


def load_trained(path: str, device: torch.device) -> nn.Module:
    \"\"\"Rebuild a trained model from a checkpoint written by ``scripts/train.py``.

    The architecture is read from the checkpoint itself, and the pretrained
    ImageNet weights are skipped, since the trained weights replace them at once.
    \"\"\"
    checkpoint = torch.load(path, map_location=device)
    model = build_model(
        encoder=checkpoint.get("encoder", "resnet34"),
        in_channels=checkpoint.get("in_channels", 2),
        weights=None,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model


@torch.no_grad()
def ensemble_predict(
    models: list[nn.Module], x: np.ndarray, tta: str = "none", **tiled_kwargs
) -> np.ndarray:
    \"\"\"Average the logits of several models, each with the same TTA applied.

    An ensemble helps when its members make different mistakes. Here the model
    trained on single-annotator labels and the one trained on the annotators'
    consensus each came out ahead on about half of the held-out validation
    images, which is exactly the situation averaging exploits. With one model
    this is simply ``tiled_predict_tta``.
    \"\"\"
    total: np.ndarray | None = None
    for model in models:
        logits = tiled_predict_tta(model, x, tta=tta, **tiled_kwargs)
        total = logits if total is None else total + logits
    return total / len(models)

""",
    'filament_seg/postprocess.py': """\"\"\"Turning a binary probability/threshold map into scored filament instances.

This is where Panoptic Quality is won or lost. The organisers name structural
continuity as challenge #3: algorithms tend to shatter one filament into a
cluster of dark "islands". Under PQ a shattered filament is one false negative
*plus* several false positives, so fragmentation is punished roughly three times
over while pixel accuracy barely moves.

The pipeline here is deliberately ordered:

``threshold`` (probability map -> binary) -> ``open`` (kill speckle) ->
``close`` (rejoin thin necks) -> ``bridge`` (join components separated by a
small gap) -> ``area filter`` (drop instances too small to plausibly clear IoU
0.5) -> ``confidence filter`` (drop instances the model is unsure of).

Every step is a tunable knob; tune them against local PQ, never against Dice.
:func:`logits_to_instances` is the single entry point used by training
validation, the post-processing sweep and the submission, so the three cannot
quietly disagree about what a setting means.
\"\"\"

from __future__ import annotations

import dataclasses
from dataclasses import dataclass
from typing import TypeVar

import cv2
import numpy as np


@dataclass
class PostprocessParams:
    \"\"\"Knobs for logits -> instances. Defaults are a starting point, not tuned.\"\"\"

    #: Probability above which a pixel counts as filament.
    threshold: float = 0.5
    open_radius: int = 2
    close_radius: int = 5
    #: Components whose gap is below this many pixels are merged into one
    #: instance. Directly targets the fragmentation penalty.
    bridge_gap: int = 12
    #: Instances smaller than this (in pixels, at 2048x2048) are dropped.
    #: A doubtful instance costs 1.0 in the PQ denominator; silence costs 0.5.
    min_area: int = 400
    #: Fill interior holes so a filament is one solid blob. Not swept: an
    #: unfilled ring can only lower IoU against the solid annotated polygons.
    fill_holes: bool = True
    #: Instances whose mean predicted probability is below this are dropped;
    #: 0 keeps everything. A prediction that misses costs 0.5 in the PQ
    #: denominator for nothing, so a faint, uncertain component is often worth
    #: more unsubmitted.
    min_confidence: float = 0.0


_Settings = TypeVar("_Settings")


def parse_settings(text: str, defaults: _Settings) -> _Settings:
    \"\"\"``"threshold=0.7,min_area=400"`` -> a copy of ``defaults`` with those fields set.

    Values are converted to each field's type; an unknown name raises
    ``ValueError`` rather than being silently ignored.
    \"\"\"
    values: dict = {}
    for item in filter(None, (part.strip() for part in text.split(","))):
        key, _, raw = item.partition("=")
        key = key.strip()
        if not hasattr(defaults, key):
            raise ValueError(f"unknown setting {key!r} for {type(defaults).__name__}")
        kind = type(getattr(defaults, key))
        values[key] = raw.strip().lower() in ("1", "true", "yes") if kind is bool else kind(raw)
    return dataclasses.replace(defaults, **values)


def probability_to_logit(p: float) -> float:
    \"\"\"The logit threshold equivalent to probability ``p`` (clamped at 0 and 1).\"\"\"
    if p <= 0.0:
        return -1e9
    if p >= 1.0:
        return 1e9
    return float(np.log(p / (1.0 - p)))


def _ellipse(radius: int) -> np.ndarray:
    size = max(1, 2 * radius + 1)
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))


def clean_binary(binary: np.ndarray, params: PostprocessParams) -> np.ndarray:
    \"\"\"Morphological cleanup of a binary mask (uint8, 0/1).\"\"\"
    out = (binary > 0).astype(np.uint8)
    if params.open_radius > 0:
        out = cv2.morphologyEx(out, cv2.MORPH_OPEN, _ellipse(params.open_radius))
    if params.close_radius > 0:
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, _ellipse(params.close_radius))
    return out


def fill_holes(binary: np.ndarray) -> np.ndarray:
    \"\"\"Fill enclosed background regions, leaving the outer background intact.\"\"\"
    mask = (binary > 0).astype(np.uint8)
    flood = mask.copy()
    h, w = mask.shape
    pad = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, pad, (0, 0), 1)
    return (mask | (1 - flood)).astype(np.uint8)


def bridge_and_label(binary: np.ndarray, gap: int) -> np.ndarray:
    \"\"\"Label connected components, merging any that lie within ``gap`` pixels.

    Dilating by ``gap/2`` and labelling *that* lets nearby fragments share a
    label; the labels are then mapped back onto the original (undilated) pixels,
    so instance shapes are unchanged -- only their grouping is.
    \"\"\"
    mask = (binary > 0).astype(np.uint8)
    if gap <= 0:
        _, labels = cv2.connectedComponents(mask, connectivity=8)
        return labels

    dilated = cv2.dilate(mask, _ellipse(max(1, gap // 2)))
    _, group_labels = cv2.connectedComponents(dilated, connectivity=8)
    return np.where(mask > 0, group_labels, 0).astype(np.int32)


def filter_by_area(labels: np.ndarray, min_area: int) -> np.ndarray:
    \"\"\"Drop instances below ``min_area`` and relabel consecutively from 1.

    One counting pass and a lookup table, rather than a full-frame comparison
    per instance: the post-processing sweep calls this thousands of times.
    \"\"\"
    counts = np.bincount(labels.ravel())
    keep = counts >= min_area
    keep[0] = False
    lookup = np.zeros(counts.size, dtype=np.int32)
    lookup[keep] = np.arange(1, int(keep.sum()) + 1, dtype=np.int32)
    return lookup[labels]


def binary_to_instances(
    binary: np.ndarray,
    params: PostprocessParams | None = None,
    restrict_to: np.ndarray | None = None,
) -> np.ndarray:
    \"\"\"Full binary -> integer label map pipeline.

    ``restrict_to`` is typically the solar disk mask: anything outside it cannot
    be a filament and is discarded before instances are formed.
    \"\"\"
    params = params or PostprocessParams()
    mask = (binary > 0).astype(np.uint8)
    if restrict_to is not None:
        mask = (mask & restrict_to.astype(np.uint8)).astype(np.uint8)

    mask = clean_binary(mask, params)
    if params.fill_holes:
        mask = fill_holes(mask)
    labels = bridge_and_label(mask, params.bridge_gap)
    return filter_by_area(labels, params.min_area)


def drop_low_confidence(
    labels: np.ndarray, logits: np.ndarray, min_confidence: float
) -> np.ndarray:
    \"\"\"Drop instances whose mean probability is below ``min_confidence``.

    Survivors are relabelled consecutively from 1, as ``filter_by_area`` does.
    Only instance pixels are passed through the sigmoid, so this stays cheap
    on a 2048x2048 map.
    \"\"\"
    if min_confidence <= 0.0:
        return labels
    inside = labels > 0
    if not inside.any():
        return labels
    ids = labels[inside]
    probs = 1.0 / (1.0 + np.exp(-logits[inside].astype(np.float64)))
    n = int(labels.max()) + 1
    sums = np.bincount(ids, weights=probs, minlength=n)
    counts = np.bincount(ids, minlength=n)
    keep = counts > 0
    keep[keep] = sums[keep] / counts[keep] >= min_confidence
    keep[0] = False
    lookup = np.zeros(n, dtype=np.int32)
    lookup[keep] = np.arange(1, int(keep.sum()) + 1, dtype=np.int32)
    return lookup[labels]


def logits_to_instances(
    logits: np.ndarray,
    disk_mask: np.ndarray,
    params: PostprocessParams | None = None,
) -> np.ndarray:
    \"\"\"Model logits -> integer instance label map, restricted to the disk.\"\"\"
    params = params or PostprocessParams()
    binary = ((logits > probability_to_logit(params.threshold)) & disk_mask).astype(np.uint8)
    labels = binary_to_instances(binary, params, restrict_to=disk_mask)
    return drop_low_confidence(labels, logits, params.min_confidence)

""",
    'filament_seg/rle.py': """\"\"\"Conversion between masks, COCO polygons and the competition's RLE format.

The submission asks for *RLE counts only* -- one row per predicted filament,
no ``size`` field, no quoting. The size is fixed at 2048x2048 for every image.

Two things bite people here:

1. ``pycocotools`` expects Fortran-ordered (column-major) arrays. Passing a
   C-ordered mask silently produces a transposed encoding that scores ~0.
2. ``counts`` comes back as ``bytes``; it must be decoded to ``str`` before it
   reaches the CSV.

The COCO counts alphabet spans ASCII 48..111 (``0``..``o``), which contains no
comma, quote or newline -- so a plain ``to_csv`` never adds quoting. We assert
this rather than trust it.
\"\"\"

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils

from .config import IMAGE_HEIGHT, IMAGE_SIZE, IMAGE_WIDTH

#: Characters that would force CSV quoting and therefore corrupt a submission.
_FORBIDDEN_IN_COUNTS = set(',"\\'\\r\\n')

Rle = dict  # {"size": [h, w], "counts": bytes}


def mask_to_rle(mask: np.ndarray) -> Rle:
    \"\"\"Encode a boolean/0-1 mask as a COCO RLE dict.\"\"\"
    if mask.dtype != np.uint8:
        mask = mask.astype(np.uint8)
    return mask_utils.encode(np.asfortranarray(mask))


def rle_to_mask(rle: Rle) -> np.ndarray:
    \"\"\"Decode a COCO RLE dict to a boolean mask.\"\"\"
    return mask_utils.decode(rle).astype(bool)


def mask_to_counts(mask: np.ndarray) -> str:
    \"\"\"Encode a mask to the bare counts string expected in the submission.\"\"\"
    return counts_of(mask_to_rle(mask))


def counts_of(rle: Rle) -> str:
    counts = rle["counts"]
    if isinstance(counts, bytes):
        counts = counts.decode("ascii")
    return counts


def rle_from_counts(
    counts: str, height: int = IMAGE_HEIGHT, width: int = IMAGE_WIDTH
) -> Rle:
    \"\"\"Rebuild a full RLE dict from a bare counts string.\"\"\"
    return {"size": [height, width], "counts": counts.encode("ascii")}


def counts_to_mask(
    counts: str, height: int = IMAGE_HEIGHT, width: int = IMAGE_WIDTH
) -> np.ndarray:
    return rle_to_mask(rle_from_counts(counts, height, width))


def polygon_to_rle(
    polygons: Sequence[Sequence[float]],
    height: int = IMAGE_HEIGHT,
    width: int = IMAGE_WIDTH,
) -> Rle:
    \"\"\"Convert a COCO ``segmentation`` field (list of flat polygons) to one RLE.

    MAGFiLO stores exactly one single-piece polygon per filament, but merging
    handles the general case without extra cost.
    \"\"\"
    rles = mask_utils.frPyObjects([list(map(float, p)) for p in polygons], height, width)
    return mask_utils.merge(rles)


def labels_to_rles(labels: np.ndarray) -> list[Rle]:
    \"\"\"Split an integer label map (0 = background) into one RLE per instance.\"\"\"
    out: list[Rle] = []
    for value in np.unique(labels):
        if value == 0:
            continue
        out.append(mask_to_rle(labels == value))
    return out


def rle_areas(rles: Sequence[Rle]) -> np.ndarray:
    if not rles:
        return np.zeros(0, dtype=np.float64)
    rles = list(rles)
    # Under NumPy 2, pycocotools' area() overflows a uint8 counter past 255
    # masks in one call, so the masks are measured in chunks of at most 255.
    return np.concatenate([np.asarray(mask_utils.area(rles[i : i + 255]), dtype=np.float64)
                           for i in range(0, len(rles), 255)])


def rle_intersection_area(a: Rle, b: Rle) -> float:
    \"\"\"Exact intersection area of two RLEs, without decoding either to a mask.\"\"\"
    return float(mask_utils.area(mask_utils.merge([a, b], intersect=1)))


def iou_matrix(gt_rles: Sequence[Rle], pred_rles: Sequence[Rle]) -> np.ndarray:
    \"\"\"Pairwise IoU with shape ``(n_gt, n_pred)``.

    Computed in C by pycocotools, so it stays cheap even at 2048x2048.
    \"\"\"
    if not gt_rles or not pred_rles:
        return np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)
    # mask_utils.iou(dt, gt, iscrowd) -> (n_dt, n_gt); transpose to (n_gt, n_pred).
    ious = mask_utils.iou(list(pred_rles), list(gt_rles), [0] * len(gt_rles))
    return np.asarray(ious, dtype=np.float64).T


# --- Submission I/O ---------------------------------------------------------


def make_filament_id(image_stem: str, index: int) -> str:
    \"\"\"Build a submission id, e.g. ``20150125172714Mh_1`` (1-based index).\"\"\"
    return f"{image_stem}_{index}"


def validate_counts(counts: str) -> None:
    bad = _FORBIDDEN_IN_COUNTS.intersection(counts)
    if bad:
        raise ValueError(f"RLE counts contain CSV-hostile characters: {sorted(bad)}")


def build_submission(
    predictions: dict[str, Sequence[Rle]],
    validate: bool = True,
) -> pd.DataFrame:
    \"\"\"Turn ``{image_stem: [rle, ...]}`` into the submission DataFrame.

    Images with no predicted filament simply contribute no rows.
    \"\"\"
    rows: list[dict[str, str]] = []
    for stem in sorted(predictions):
        for i, rle in enumerate(predictions[stem], start=1):
            counts = counts_of(rle)
            if validate:
                validate_counts(counts)
            rows.append(
                {"filament_id": make_filament_id(stem, i), "segmentation_rle": counts}
            )
    frame = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    if frame["filament_id"].duplicated().any():
        raise ValueError("duplicate filament_id values in submission")
    return frame


def write_submission(frame: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def read_submission(path: str | Path) -> dict[str, list[Rle]]:
    \"\"\"Read a submission CSV back into ``{image_stem: [rle, ...]}``.\"\"\"
    frame = pd.read_csv(path, dtype=str).fillna("")
    grouped: dict[str, list[Rle]] = {}
    for filament_id, counts in zip(frame["filament_id"], frame["segmentation_rle"]):
        stem = filament_id.rsplit("_", 1)[0]
        grouped.setdefault(stem, []).append(rle_from_counts(counts, *IMAGE_SIZE))
    return grouped


def iter_masks(rles: Iterable[Rle]) -> Iterable[np.ndarray]:
    for rle in rles:
        yield rle_to_mask(rle)

""",
    'filament_seg/scoring.py': """\"\"\"Pooled Panoptic Quality totals and the paired bootstrap that compares them.

The competition pools TP, FP, FN and the matched IoU sum over every annotator's
view of every observation. Keeping those six totals per observation, rather
than a per-observation PQ, is what lets comparisons resample observations and
still compute the pooled score exactly.
\"\"\"

from __future__ import annotations

from typing import Sequence

import numpy as np

from .metrics import evaluate_image
from .rle import Rle

#: Per-observation totals, in this order, summed over the observation's views.
COUNT_FIELDS = ("n_gt", "n_pred", "tp", "fp", "fn", "iou_sum")


def view_totals(views: Sequence[str], gt: dict[str, list[Rle]], pred_rles: list[Rle]) -> np.ndarray:
    \"\"\"One observation's totals: its predictions scored against each view, summed.\"\"\"
    out = np.zeros(len(COUNT_FIELDS), dtype=np.float64)
    for image_id in views:
        r = evaluate_image(image_id, gt[image_id], pred_rles, with_fragmentation=False)
        out += (r.n_gt, r.n_pred, r.tp, r.fp, r.fn, r.iou_sum)
    return out


def pq_of(totals: np.ndarray) -> np.ndarray:
    \"\"\"Pooled PQ from summed totals; works on any leading shape.\"\"\"
    tp, fp, fn, iou_sum = (totals[..., COUNT_FIELDS.index(k)] for k in ("tp", "fp", "fn", "iou_sum"))
    denominator = tp + 0.5 * fp + 0.5 * fn
    return np.divide(iou_sum, denominator, out=np.zeros_like(iou_sum), where=denominator > 0)


def paired_bootstrap(
    per_stem_a: np.ndarray, per_stem_b: np.ndarray, n_boot: int = 2000, seed: int = 0
) -> tuple[float, float, float]:
    \"\"\"PQ(a) - PQ(b) on all observations, with a 95% interval from resampling them.\"\"\"
    deltas = bootstrap_deltas(per_stem_a, per_stem_b, n_boot, seed)
    full = float(pq_of(per_stem_a.sum(axis=0)) - pq_of(per_stem_b.sum(axis=0)))
    low, high = np.percentile(deltas, [2.5, 97.5])
    return full, float(low), float(high)


def bootstrap_deltas(
    per_stem_a: np.ndarray, per_stem_b: np.ndarray, n_boot: int = 2000, seed: int = 0
) -> np.ndarray:
    \"\"\"PQ(a) - PQ(b) on ``n_boot`` resamples of the observations.

    Observations, not views, are resampled -- an observation's views share one
    set of predictions -- and both settings see the same resample each time.
    \"\"\"
    n = per_stem_a.shape[0]
    weights = np.random.default_rng(seed).multinomial(n, np.full(n, 1.0 / n), size=n_boot)
    # einsum rather than `@`: macOS's Accelerate BLAS raises spurious
    # floating-point warnings in matmul, and this is far too small to need BLAS.
    return (pq_of(np.einsum("bs,sk->bk", weights, per_stem_a))
            - pq_of(np.einsum("bs,sk->bk", weights, per_stem_b)))

""",
    'official_pq.py': """\"\"\"Official-notebook-compatible pooled PQ. No model or dataset downloads.

Reference: azimahmadzadeh/self-evaluation-notebook, version 6, retrieved
2026-11-02. The reference counts ALL IoU > 0.5 pairs, not Hungarian matches.
This module deliberately preserves that behavior, including overlapping GT.
\"\"\"
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Mapping, Sequence

import numpy as np
from pycocotools import mask as coco_mask


@dataclass
class Totals:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    iou_sum: float = 0.0
    ambiguous_gt: int = 0
    ambiguous_pred: int = 0
    fragmented_gt: int = 0
    merged_pred: int = 0

    def __add__(self, other: Totals) -> Totals:
        return Totals(**{k: v + asdict(other)[k] for k, v in asdict(self).items()})

    def summary(self) -> dict:
        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return {
            **asdict(self),
            "pq": self.iou_sum / denominator if denominator else 0.0,
            "dq": self.tp / denominator if denominator else 0.0,
            "sq": self.iou_sum / self.tp if self.tp else 0.0,
        }


def from_iou(ious: np.ndarray) -> Totals:
    \"\"\"Count exactly as the host notebook; diagnostics use any positive IoU.\"\"\"
    ious = np.asarray(ious, dtype=np.float64)
    if ious.ndim != 2 or not np.isfinite(ious).all():
        raise ValueError("IoU matrix must be a finite 2-D array")
    if ((ious < 0) | (ious > 1)).any():
        raise ValueError("IoU values must be in [0, 1]")
    hit = ious > 0.5
    links = ious > 0
    return Totals(
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=0) == 0).sum()),
        fn=int((hit.sum(axis=1) == 0).sum()),
        iou_sum=float(ious[hit].sum()),
        ambiguous_gt=int((hit.sum(axis=1) > 1).sum()),
        ambiguous_pred=int((hit.sum(axis=0) > 1).sum()),
        fragmented_gt=int((links.sum(axis=1) > 1).sum()),
        merged_pred=int((links.sum(axis=0) > 1).sum()),
    )


def canonical_rle(rle: dict) -> dict:
    size = list(map(int, rle["size"]))
    if len(size) != 2 or min(size) <= 0:
        raise ValueError("RLE size must be positive height,width")
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    if isinstance(counts, list):
        return coco_mask.frPyObjects({"size": size, "counts": counts}, *size)
    return {"size": size, "counts": counts}


def pairwise_iou(gt: Sequence[dict], pred: Sequence[dict]) -> np.ndarray:
    if not gt or not pred:
        return np.zeros((len(gt), len(pred)), dtype=np.float64)
    gt, pred = list(map(canonical_rle, gt)), list(map(canonical_rle, pred))
    shapes = {tuple(r["size"]) for r in gt + pred}
    if len(shapes) != 1:
        raise ValueError("GT and prediction RLE sizes differ; resize before encoding")
    return np.asarray(coco_mask.iou(pred, gt, [0] * len(gt)), dtype=np.float64).T


def require_disjoint(pred: Sequence[dict], shape=(2048, 2048)) -> None:
    \"\"\"Reject invalid submissions rather than silently changing their masks.\"\"\"
    occupied = np.zeros(shape, dtype=bool)
    for r in pred:
        r = canonical_rle(r)
        if tuple(r["size"]) != tuple(shape):
            raise ValueError("Prediction has unexpected canvas size")
        current = coco_mask.decode(r).astype(bool)
        if not current.any():
            raise ValueError("Empty mask is not a filament instance")
        if (occupied & current).any():
            raise ValueError("Prediction instances overlap")
        occupied |= current


def evaluate_records(
    records: Mapping[str, tuple[str, Sequence[dict]]],
    predictions: Mapping[str, Sequence[dict]],
    *, validate_predictions: bool = True, shape=(2048, 2048),
) -> dict:
    \"\"\"One prediction set per photo, scored against EVERY annotator separately.

    records: {annotator_record_id: (physical_photo_stem, list_of_GT_RLEs)}.
    Missing predictions are [], and unknown prediction photos are rejected.
    Explicit zero-GT records are supported using the host score's empty-row
    rule. Unlike its CSV-only entry discovery, the manifest retains such rows.
    \"\"\"
    known = {stem for stem, _ in records.values()}
    unknown = set(predictions) - known
    if unknown:
        raise ValueError(f"Predictions outside evaluation split: {sorted(unknown)[:3]}")
    for stem in known:
        if validate_predictions:
            require_disjoint(predictions.get(stem, []), shape)
    total = Totals()
    by_photo: dict[str, Totals] = {}
    by_record = {}
    nonzero_ious, matched_ious = [], []
    for record_id, (stem, gt) in records.items():
        pred = predictions.get(stem, [])
        for r in list(gt) + list(pred):
            if tuple(r["size"]) != tuple(shape):
                raise ValueError("RLE canvas differs from evaluation canvas")
        ious = pairwise_iou(gt, pred)
        result = from_iou(ious)
        total = total + result
        by_photo[stem] = by_photo.get(stem, Totals()) + result
        by_record[record_id] = result.summary()
        nonzero_ious.extend(ious[ious > 0].tolist())
        matched_ious.extend(ious[ious > 0.5].tolist())
    return {
        "pooled": total.summary(),
        "n_photos": len(known), "n_annotator_records": len(records),
        "n_prediction_instances_physical": sum(len(predictions.get(s, [])) for s in known),
        "per_photo": {s: t.summary() for s, t in sorted(by_photo.items())},
        "per_record": by_record,
        "positive_pair_ious": nonzero_ious,
        "positive_pair_dices": [2 * x / (1 + x) for x in nonzero_ious],
        "matched_pair_ious": matched_ious,
        "notes": "Pooled all-pairs host semantics; no one-to-one assignment. Empty total returns 0.",
    }
""",
    'predict.py': """\"\"\"GitHub-native inference and Blended mosaic projection.\"\"\"
import cv2
import numpy as np
import torch

from filament_seg.disk import detect_disk, flatten, read_image
from filament_seg.model import load_trained, tiled_predict_tta
from filament_seg.rle import counts_of, mask_to_rle

from recovery import item


class Models:
    def __init__(self, weights, sizes, blended_source, blended):
        from ultralytics import YOLO

        self.device = torch.device("cuda:0")
        self.unet = load_trained(str(weights[0]), self.device)
        self.detectors = [(YOLO(str(path)), size, path.name) for path, size in zip(weights[1:], sizes)]
        self.source = blended_source
        self.blended = blended

    @torch.inference_mode()
    def predict(self, path):
        image = read_image(path)
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image.shape != (2048, 2048) or gray is None or gray.shape != image.shape:
            raise ValueError(f"Unexpected native image shape for {path.name}")
        disk = detect_disk(image)
        flat = (np.clip(flatten(image, disk), 0, 2) * 127.5).astype(np.uint8)
        x = np.stack([flat.astype(np.float32) / 255, disk.radius_map(image.shape).astype(np.float32)])
        logits = tiled_predict_tta(self.unet, x, tta="dihedral", tile=512,
                                   overlap=128, device=self.device).astype(np.float16).astype(np.float32)
        pb = self.source.mosaic_predict(self.blended, gray).cpu().numpy()
        if pb.shape != gray.shape or not np.isfinite(pb).all() or not np.isfinite(logits).all():
            raise ValueError(f"Invalid semantic model output on {path.name}")
        raw, views = [], []
        from filament_seg.fusion import Detection

        for detector, size, name in self.detectors:
            result = detector.predict(cv2.cvtColor(flat, cv2.COLOR_GRAY2BGR),
                                      imgsz=size, conf=.05, iou=.6, max_det=150,
                                      retina_masks=True, device=0, verbose=False)[0]
            view = []
            if result.masks is not None:
                masks = result.masks.data.cpu().numpy()
                if masks.shape[1:] != gray.shape:
                    raise RuntimeError("YOLO mask was not mapped back to the native canvas")
                for n, (mask, score, box) in enumerate(zip(masks, result.boxes.conf.cpu().tolist(),
                                                         result.boxes.xyxy.cpu().tolist())):
                    if mask.any():
                        rle = mask_to_rle(mask)
                        score = round(score, 4)
                        raw.append(item(rle, score, "yolo", f"{size}-{n}", model=name,
                                        view="native", box=box))
                        view.append(Detection(score, counts_of(rle)))
            views.append(view)
        return gray, disk.mask(image.shape), logits, pb, raw, views
""",
    'recovery.py': """\"\"\"Native-space probability fusion and conservative instance recovery.\"\"\"
from dataclasses import dataclass

import cv2
import numpy as np
from pycocotools import mask as cm

from filament_seg.rle import mask_to_rle, rle_to_mask


@dataclass(frozen=True)
class Rules:
    high: float = 0.48
    low: float = 0.30
    weight: float = 0.55
    feather: int = 64
    min_area: int = 64
    detection_score: float = 0.30
    duplicate_iou: float = 0.55
    grow: int = 6


def blend(pg, pb, center, weight, feather):
    \"\"\"Keep Pg unchanged wherever Pb has no support.\"\"\"
    start, end = center
    size = end - start
    if pg.shape != pb.shape or not (0 <= start < end <= min(pg.shape)):
        raise ValueError("Invalid probability geometry")
    distance = np.minimum(np.arange(size), np.arange(size)[::-1])
    ramp = np.minimum(distance / feather, 1) if feather else np.ones(size)
    alpha = (weight * np.minimum.outer(ramp, ramp)).astype(np.float32)
    region = np.s_[start:end, start:end]
    result = pg.copy()
    result[region] = pg[region] + alpha * (pb[region] - pg[region])
    return result


def components(binary):
    _, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary.astype(np.uint8), connectivity=8
    )
    return labels, stats


def dilation(mask, radius):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1,) * 2)
    return cv2.dilate(mask.astype(np.uint8), kernel).astype(bool)


def item(rle, score, source, key, **details):
    return dict(rle=rle, score=float(score), source=source, key=key, **details)


def supported_candidates(detections, prob, disk, rules):
    \"\"\"Connect masks to semantic regions without joining instance identities.\"\"\"
    labels, stats = components((prob >= rules.high) & disk)
    candidates, audit = [], []
    for det in detections:
        x, y, w, h = map(int, cm.toBbox(det["rle"]))
        padding = rules.grow + 2
        roi = np.s_[max(0, y - padding):y + h + padding,
                    max(0, x - padding):x + w + padding]
        raw = rle_to_mask(det["rle"])[roi].astype(bool) & disk[roi]
        if not raw.any():
            continue
        grown = dilation(raw, rules.grow) & disk[roi]
        local_labels = labels[roi]
        local_prob = prob[roi]
        edges = []
        area = int(raw.sum())
        for index in np.unique(local_labels[grown]):
            if not index:
                continue
            x, y, w, h, size = map(int, stats[index])
            segment = local_labels == index
            overlap = int((segment & raw).sum())
            near = int((segment & grown).sum())
            coverage = overlap / size
            edges.append(dict(component=int(index), iou=overlap / (area + size - overlap),
                              coverage=coverage, support=overlap / area,
                              grown_coverage=near / size, bbox=[x, y, w, h],
                              linked=overlap >= 8 or near >= max(16, 0.15 * size)))
        linked = [e["component"] for e in edges if e["linked"]]
        seed = grown & np.isin(local_labels, linked)
        guided = raw
        if seed.any():
            # Retain only low-threshold components connected to a high seed.
            local, _ = components(grown & (local_prob >= rules.low))
            selected = np.setdiff1d(np.unique(local[seed]), [0])
            proposed = np.isin(local, selected)
            if proposed.sum() >= max(rules.min_area, 0.40 * area):
                guided = proposed
        support = float(local_prob[raw].mean())
        accepted = (det["score"] >= rules.detection_score
                    and (support >= 0.14 or det["score"] >= 0.70))
        log = dict(key=det["key"], edges=edges, accepted=accepted, support=support,
                   linked=linked, raw_score=det["score"])
        audit.append(log)
        if accepted:
            canvas = np.zeros_like(disk)
            canvas[roi] = guided
            candidates.append(item(mask_to_rle(canvas), 0.75 * det["score"] + 0.25 * support,
                                   "yolo", det["key"], raw_rle=det["rle"], linked=linked))
    return candidates, labels, audit


def finalize(candidates, shape, rules):
    \"\"\"Suppress high-IoU duplicates, then assign shared pixels by priority.\"\"\"
    kept, audit = [], []
    claimed = np.zeros(shape, dtype=bool)
    for candidate in sorted(candidates, key=lambda d: (d["source"] != "yolo", -d["score"], d["key"])):
        rle = candidate["rle"]
        old = [other["rle"] for other in kept]
        if old and cm.iou([rle], old, [0] * len(old)).max() > rules.duplicate_iou:
            audit.append((candidate["key"], "duplicate"))
            continue
        mask = rle_to_mask(rle).astype(bool)
        owned = mask & ~claimed
        if owned.sum() < max(rules.min_area, 0.35 * mask.sum()):
            audit.append((candidate["key"], "insufficient_owned_area"))
            continue
        claimed |= owned
        kept.append({**candidate, "rle": mask_to_rle(owned)})
        audit.append((candidate["key"], "kept"))
    return kept, claimed, audit


def skeleton_length(region):
    \"\"\"Approximate centerline length with an iterative morphological skeleton.\"\"\"
    layer = np.pad(region.astype(np.uint8), 1)
    skeleton = np.zeros_like(layer)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    while layer.any():
        eroded = cv2.erode(layer, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0)
        skeleton |= layer & ~cv2.dilate(eroded, kernel)
        layer = eroded
    return int(skeleton.sum())


def orphans(labels, claimed, pg, pb, prob, valid, gray, rules):
    \"\"\"Grade whole or residual regions with local morphology and branch support.\"\"\"
    residual, stats = components((labels > 0) & ~claimed)
    candidates, audit = [], []
    for index, (x, y, w, h, area) in enumerate(stats[1:], 1):
        area = int(area)
        if area < rules.min_area:
            continue
        window = np.s_[max(0, y - 8):y + h + 8, max(0, x - 8):x + w + 8]
        region = residual[window] == index
        native = valid[window][region]
        parent_ids = np.unique(labels[window][region])
        ring = dilation(region, 7) & ~region
        parent_contact = float((ring & claimed[window]).sum() / max(1, ring.sum()))
        skeleton = skeleton_length(region)
        contrast = float(gray[window][ring].mean() - gray[window][region].mean()) if ring.any() else 0.
        g = float(pg[window][region].mean())
        b = float(pb[window][region][native].mean()) if native.any() else None
        dual = float((np.minimum(pg[window][region], pb[window][region])[native] >= .34).mean()) if native.any() else None
        score = float(prob[window][region].mean())
        small = area < 250
        has_evidence = g >= .48 and (dual >= .55 if dual is not None else g >= .68)
        plausible = (skeleton >= 12 and skeleton ** 2 / area >= 1.8
                     and contrast >= (4 if small else 2) and parent_contact <= .16)
        grade = ("high_confidence" if has_evidence and plausible and score >= (.65 if small else .55)
                 else "review" if plausible or has_evidence else "reject")
        key = f"semantic-{index}"
        audit.append(dict(key=key, grade=grade, area=area, pg=g, pb=b, dual=dual,
                          valid_fraction=float(native.mean()), score=score, skeleton=skeleton,
                          contrast=contrast, contact=parent_contact, parents=parent_ids.tolist()))
        if grade == "high_confidence":
            mask = np.zeros_like(claimed)
            mask[window] = region
            candidates.append(item(mask_to_rle(mask), score, "semantic", key))
    return candidates, audit
""",
    'submission_audit.py': """\"\"\"Inference-only handoff and strict CSV audit for the frozen V2 pipeline.\"\"\"
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd
from pycocotools import mask as cm


def discover_experiment(explicit='',experiment='V2-Full',input_root='/kaggle/input',working_root='/kaggle/working'):
    def eligible(p):return all((p/n).is_file() for n in ['config.json','normalization.json','manifest.json','run_summary.json','calibration/thresholds.json'])
    if explicit:
        p=Path(explicit)
        if not eligible(p):raise ValueError('EXPERIMENT_SOURCE must contain a completed calibrated experiment, including calibration/thresholds.json')
        return p.resolve()
    local=[Path(working_root)/folder/experiment for folder in ['solar_v2_resume','solar_v2']]
    candidates=[p.resolve() for p in local if eligible(p)]
    if not candidates:
        candidates=sorted({p.parent.resolve() for p in Path(input_root).rglob('config.json') if p.parent.name==experiment and eligible(p.parent)})
    if len(candidates)!=1:raise ValueError(f'Found {len(candidates)} completed {experiment} experiments: {[str(p) for p in candidates]}. Set EXPERIMENT_SOURCE explicitly.')
    return candidates[0]


def audit_csv(path,test_ids,coverage=None,shape=(2048,2048)):
    path=Path(path)
    if path.suffix.lower()!='.csv':raise ValueError('Submit submission.csv, never .pyc/.py/ipynb or an ablation-results table')
    frame=pd.read_csv(path,dtype=str,keep_default_na=False)
    if list(frame.columns)!=['filament_id','segmentation_rle']:raise ValueError('Wrong CSV columns; this may be an experiment-results table rather than a submission')
    if frame.empty or frame.filament_id.duplicated().any():raise ValueError('Empty submission or duplicate filament ids')
    known=set(test_ids);groups={};counts={s:0 for s in known}
    for row in frame.itertuples(index=False):
        match=re.fullmatch(r'(.+)_([1-9][0-9]*)',row.filament_id)
        if not match or match[1] not in known:raise ValueError(f'Invalid test image/instance id: {row.filament_id}')
        if not row.segmentation_rle:raise ValueError('Empty RLE')
        groups.setdefault(match[1],[]).append((int(match[2]),row.segmentation_rle));counts[match[1]]+=1
    for stem,rows in groups.items():
        if sorted(i for i,_ in rows)!=list(range(1,len(rows)+1)):raise ValueError('Instance ids must be consecutive from 1')
        occupied=np.zeros(shape,bool)
        for _,encoded in rows:
            validate_rle_canvas(encoded,int(np.prod(shape)))
            try:mask=cm.decode({'size':list(shape),'counts':encoded.encode('ascii')}).astype(bool)
            except Exception as error:raise ValueError(f'Malformed RLE for {stem}') from error
            if mask.shape!=shape or not mask.any():raise ValueError('Unexpected mask shape or empty mask')
            if (occupied&mask).any():raise ValueError(f'Overlapping instances: {stem}')
            occupied|=mask
    if coverage is not None:
        if not coverage.get('all_test_photos_processed') or coverage.get('counts_by_photo')!=counts:
            raise ValueError('Test coverage does not match generated CSV')
    return {'validated':True,'columns':list(frame.columns),'n_rows':len(frame),'n_test_photos':len(known),
        'photos_with_predictions':len(groups),'empty_prediction_photos':sorted(known-set(groups)),
        'shape':list(shape),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}


def validate_rle_canvas(encoded,pixels):
    \"\"\"Decode COCO's signed/delta run lengths before calling the C mask decoder.\"\"\"
    counts=[];position=0;total=0
    while position<len(encoded):
        value=0;shift=0
        while True:
            if position>=len(encoded):raise ValueError('Truncated COCO RLE')
            char=ord(encoded[position])-48;position+=1
            if not 0<=char<=63 or shift>35:raise ValueError('Invalid compressed COCO RLE')
            value|=(char&31)<<shift;shift+=5
            if not char&32:
                if char&16:value|=-(1<<shift)
                break
        if len(counts)>2:value+=counts[-2]
        if value<0:raise ValueError('Negative RLE run length')
        counts.append(value);total+=value
        if total>pixels:raise ValueError('RLE canvas exceeds expected image size')
    if total!=pixels:raise ValueError('RLE canvas does not match expected 2048x2048 image size')

""",
    'train_blended.py': """\"\"\"Train the supplied Blended architecture on the GitHub training observations.\"\"\"
import json
import math
import random
import time

import numpy as np
import torch


def fit(source, annotations, train_dir, train_ids, scratch, output, signature, batch):
    device = torch.device(source.DEVICE)
    use_cuda = device.type == "cuda"
    random.seed(source.SEED)
    np.random.seed(source.SEED)
    torch.manual_seed(source.SEED)
    scratch.mkdir(parents=True, exist_ok=True)
    ids = set(train_ids)
    raw = json.loads(annotations.read_text(encoding="utf-8"))
    filtered = {
        **raw,
        "images": [im for im in raw["images"] if str(im["id"]) in ids],
        "annotations": [a for a in raw["annotations"] if str(a["image_id"]) in ids],
    }
    expected = {im["file_name"] for im in filtered["images"]}
    filtered_path = scratch / "train_only.json"
    filtered_path.write_text(json.dumps(filtered), encoding="utf-8")
    source.ANN_JSON = str(filtered_path)
    source.TRAIN_DIR = str(train_dir)
    source.CACHE_DIR = str(scratch / "cache")
    # Resolve pretrained initialization before building a multi-GB cache.
    print("[Blended] initializing encoder", flush=True)
    model = source.UNet(pretrained=True).to(device, memory_format=torch.channels_last)
    images, masks, points, counts, names = source.build_cache()
    if set(names) != expected:
        raise RuntimeError(f"Blended cache differs from the training split: {len(names)} vs {len(expected)}")
    norm = source.build_norm_cache(images, len(names))
    if norm.shape != (len(names), 2) or not np.isfinite(norm).all() or (norm[:, 1] <= 0).any():
        raise RuntimeError("Invalid cached Blended intensity statistics")
    dataset = source.FilamentCrops(images, masks, points, counts, norm, np.arange(len(names)))
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch, shuffle=True,
                                         num_workers=2 if use_cuda else 0, pin_memory=use_cuda,
                                         persistent_workers=use_cuda)
    opt = torch.optim.AdamW(source.build_param_groups(model))
    warmup = source.WARMUP_EPOCHS * len(loader)
    steps = source.EPOCHS * len(loader)
    schedule = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda step: (step + 1) / max(1, warmup) if step < warmup else
        0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
    )
    scaler = torch.amp.GradScaler(device.type, enabled=use_cuda)
    pos_weight = torch.tensor([source.POS_WEIGHT], device=source.DEVICE)
    for epoch in range(source.EPOCHS):
        start = time.perf_counter()
        model.train()
        running = torch.zeros((), device=source.DEVICE)
        for step, (x, target) in enumerate(loader, 1):
            x = x.to(source.DEVICE, non_blocking=True, memory_format=torch.channels_last)
            target = target.to(source.DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_cuda):
                loss = source.criterion(model(x), target, pos_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
            scaler.step(opt)
            scaler.update()
            schedule.step()
            running += loss.detach()
            if step == 1 or step % 100 == 0:
                print(f"epoch {epoch + 1}/{source.EPOCHS}, step {step}/{len(loader)}, "
                      f"loss {float(running / step):.4f}", flush=True)
        mean_loss = float(running / len(loader))
        if not math.isfinite(mean_loss):
            raise RuntimeError("Nonfinite Blended training loss")
        row = dict(epoch=epoch + 1, loss=mean_loss, seconds=time.perf_counter() - start,
                   peak_gpu_bytes=torch.cuda.max_memory_allocated(device) if use_cuda else 0)
        with (output.parent / "blended_training.jsonl").open("a", encoding="utf-8") as log:
            log.write(json.dumps(row) + "\\n")
        print(row, flush=True)
    temporary = output.with_suffix(".tmp")
    torch.save(dict(model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                    signature=signature, epochs=source.EPOCHS), temporary)
    temporary.replace(output)
    return model.eval()


def restore(source, path, signature):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["signature"] != signature or checkpoint["epochs"] != source.EPOCHS:
        raise ValueError("Blended checkpoint does not match the split, source and recipe")
    model = source.UNet(pretrained=False)
    model.load_state_dict(checkpoint["model"], strict=True)
    return model.to(source.DEVICE, memory_format=torch.channels_last).eval()
""",
}

assert CONFIG["mode"] in ("preflight", "train", "smoke", "eval", "submit", "full")
dependencies = {"pycocotools": "pycocotools==2.0.10"}
if CONFIG["mode"] != "preflight":
    dependencies.update({"segmentation_models_pytorch": "segmentation-models-pytorch==0.5.0",
                         "ultralytics": "ultralytics==8.4.173", "timm": "timm==1.0.29",
                         "albumentations": "albumentations==2.0.8", "tqdm": "tqdm==4.67.1"})
missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    if not CONFIG["install_missing"]:
        raise RuntimeError(f"Attach/install missing dependencies: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
base = Path(CONFIG["scratch_root"]) / "source"
base.mkdir(parents=True, exist_ok=True)
for name, content in MODULES.items():
    path = base / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
entry = "import json, sys; from driver import run; run(json.loads(sys.argv[1]))"
env = dict(os.environ, PYTHONUNBUFFERED="1", YOLO_AUTOINSTALL="false")
subprocess.run([sys.executable, "-c", entry, json.dumps(CONFIG)], cwd=base, env=env, check=True)
